# QFID-CXR — Notebook 1/3
## Matched NIH whole-image + mined-patch training + frozen CheXzero reference

This version preserves the original notebook architecture and protocol, with only these requested changes:

1. CheXpert target pre-adaptation and source replay are removed.
2. The NIH cohort is the official **exactly-one-positive-of-five cohort of 20,746 images**.
3. Patch inputs are only `stage3_causal_complete.pt`, `stage3_spur_in_complete.pt`, and `stage3_spur_out_complete.pt`.

Everything else is retained: independent ImageNet ViT-B/16 whole-image and patch branches, final-two-block training, disentanglement head, losses, patient-disjoint NIH validation, frozen CheXzero reference/cache, resume checkpoints, audit reports, and the Stage-1 bundle.


## 1. Environment and configuration
### 1.1 Imports, CUDA policy, and fixed pathology order


In [1]:
from __future__ import annotations
import os, re, json, math, time, copy, random, hashlib, pickle, warnings, traceback
import importlib.util, subprocess, sys

# Kaggle does not include open_clip by default. Install it immediately so a missing
# dependency cannot terminate the run after the expensive dataset-preparation stage.
if importlib.util.find_spec("open_clip") is None:
    print("[SETUP] Installing required CheXzero dependency: open_clip_torch ...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "open_clip_torch"])
import open_clip
print(f"[SETUP] open_clip ready (version={getattr(open_clip, '__version__', 'unknown')})", flush=True)

from contextlib import contextmanager
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU required. Enable T4 x2 (or another CUDA accelerator).")
DEVICE=torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True

CLASSES=("Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion")
N_CLASSES=len(CLASSES)
LABEL_COLS=tuple(f"label_{i}" for i in range(N_CLASSES))
PIPELINE_VERSION="stage1-nih-exact-one-complete-packs-no-chexpert-v12"

@dataclass
class CFG:
    seed:int=42
    output_root:str="/kaggle/working/qfid_matched_stage1_nih_only"

    # Execution mode: keep "smoke" for the first end-to-end validation run.
    # Change ONLY this field to "full" for the real experiment.
    run_mode:str="full"  # "smoke" | "full"
    smoke_nih_images:int=20
    smoke_nih_epochs:int=1
    smoke_candidate_multiplier:int=4

    # Existing GPU-19-Aug inputs
    nih_stage_root:str="/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder"
    exact_one_csv:str="/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/nih_exactly_one_of_five_full_cohort.csv"
    expected_cohort_size:int=20746
    nih_image_root:str="/kaggle/input/datasets/nih-chest-xrays"
    nih_official_csv:str="/kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv"
    chexzero_checkpoint:str="/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/best_64_5e-05_original_22000_0.864.pt"

    # Preserved representation training values from GPU-19-Aug
    nih_epochs:int=10
    unfreeze_blocks:int=2
    backbone_lr:float=1e-5
    head_lr:float=3e-4
    weight_decay:float=1e-2
    batch_images:int=16
    num_workers:int=2
    amp:bool=True
    target_patch_count:int=8
    source_auc_rank_weight:float=0.05
    orth_weight:float=0.02

    # Cohorts / engineering
    nih_val_fraction:float=0.20
    runtime_limit_hours:float=9.5
    runtime_guard_minutes:float=18.0
    cache_chexzero:bool=True
    use_data_parallel:bool=True
    resume:bool=True

    # Logging: smoke logs every training/eval batch; full mode logs periodically.
    smoke_log_every_batches:int=1
    full_log_every_batches:int=25
    smoke_log_every_images:int=10
    full_log_every_images:int=1000

cfg=CFG()
if cfg.run_mode not in {"smoke","full"}:
    raise ValueError("cfg.run_mode must be either 'smoke' or 'full'.")

def smoke_enabled(): return cfg.run_mode == "smoke"
def effective_nih_epochs(): return cfg.smoke_nih_epochs if smoke_enabled() else cfg.nih_epochs
def effective_num_workers(): return 0 if smoke_enabled() else cfg.num_workers
def batch_log_every(): return cfg.smoke_log_every_batches if smoke_enabled() else cfg.full_log_every_batches
def image_log_every(): return cfg.smoke_log_every_images if smoke_enabled() else cfg.full_log_every_images

def data_parallel_enabled():
    # Avoid DP replication overhead in the tiny smoke test; use both T4s in full mode when requested.
    return bool(cfg.use_data_parallel and (not smoke_enabled()) and torch.cuda.device_count()>1)

# Smoke outputs are physically isolated from both old smoke runs and the full run.
base_out=Path(cfg.output_root)
OUT=(base_out/f"smoke_{cfg.smoke_nih_images}nih_v12") if smoke_enabled() else base_out
OUT.mkdir(parents=True,exist_ok=True)
START_TIME=time.time()
LOG_PATH=OUT/"execution_trace.log"
RUN_EVENTS=[]

def gpu_snapshot():
    if not torch.cuda.is_available(): return {}
    snap={}
    for i in range(torch.cuda.device_count()):
        snap[f"gpu{i}_alloc_mb"]=round(torch.cuda.memory_allocated(i)/(1024**2),1)
        snap[f"gpu{i}_reserved_mb"]=round(torch.cuda.memory_reserved(i)/(1024**2),1)
    return snap

def log_event(message, level="INFO", step=None, **fields):
    elapsed=time.time()-START_TIME
    event={"elapsed_s":round(elapsed,3),"level":str(level),"step":step,"message":str(message),**fields}
    RUN_EVENTS.append(event)
    prefix=f"[+{elapsed:8.1f}s] [{str(level).upper():7s}]"
    if step: prefix+=f" [{step}]"
    suffix=""
    if fields:
        suffix=" | "+" | ".join(f"{k}={v}" for k,v in fields.items())
    line=f"{prefix} {message}{suffix}"
    print(line,flush=True)
    try:
        with LOG_PATH.open("a",encoding="utf-8") as f:
            f.write(line+"\n")
    except Exception:
        pass

def cell_begin(section,title):
    log_event(f"BEGIN CELL — {title}",level="CELL",step=section)
    return time.time()

def cell_end(t0,section,title):
    log_event(f"END CELL — {title}",level="CELL",step=section,elapsed=f"{time.time()-t0:.2f}s")

@contextmanager
def timed_step(step,message,**fields):
    t0=time.time(); log_event(f"START — {message}",level="STEP",step=step,**fields)
    try:
        yield
    except Exception as e:
        log_event(f"FAILED — {message}: {type(e).__name__}: {e}",level="ERROR",step=step,
                  elapsed=f"{time.time()-t0:.2f}s",**gpu_snapshot())
        raise
    else:
        log_event(f"DONE — {message}",level="OK",step=step,elapsed=f"{time.time()-t0:.2f}s",**gpu_snapshot())

def progress_log(step,label,current,total,**fields):
    total=max(1,int(total)); current=int(current)
    log_event(f"{label} {current}/{total} ({100*current/total:.1f}%)",level="PROGRESS",step=step,**fields)

# New-run separator; keep append semantics for exact resume visibility.
log_event("="*72,level="RUN")
log_event("QFID-CXR Notebook 1 started",level="RUN",step="1.1",pipeline_version=PIPELINE_VERSION,run_mode=cfg.run_mode)
log_event("CUDA inventory",level="INFO",step="1.1",gpus=torch.cuda.device_count(),
          names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
log_event("Execution configuration",level="INFO",step="1.1",output=str(OUT),nih_epochs=effective_nih_epochs(),
          workers=effective_num_workers(),data_parallel=data_parallel_enabled())
if smoke_enabled():
    log_event("SMOKE TEST ONLY — diagnostic outputs must not be used as scientific results.",level="WARN",step="1.1",
              nih_cap=cfg.smoke_nih_images)


[SETUP] Installing required CheXzero dependency: open_clip_torch ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
[SETUP] open_clip ready (version=3.3.0)
[+     0.0s] [RUN    ] ========================================================================
[+     0.0s] [RUN    ] [1.1] QFID-CXR Notebook 1 started | pipeline_version=stage1-nih-exact-one-complete-packs-no-chexpert-v12 | run_mode=full
[+     0.0s] [INFO   ] [1.1] CUDA inventory | gpus=2 | names=['Tesla T4', 'Tesla T4']
[+     0.0s] [INFO   ] [1.1] Execution configuration | output=/kaggle/working/qfid_matched_stage1_nih_only | nih_epochs=10 | workers=2 | data_parallel=True


### 1.2 Smoke-test controls and output isolation

Smoke mode uses a deterministic 20-image NIH subset and one NIH epoch. Full mode uses the verified 20,746-image official cohort. Smoke outputs remain isolated from full-run checkpoints.


In [2]:
_CELL_T0=cell_begin("1.2","Smoke-test controls and output isolation")
def stable_take(df,n,salt):
    if n is None or n <= 0 or len(df) <= n:
        return df.copy().reset_index(drop=True)
    q=df.copy()
    key_col="image_id" if "image_id" in q.columns else ("Path" if "Path" in q.columns else "patient_id")
    q["_smoke_key"]=q[key_col].astype(str).map(lambda x: hashlib.sha256(f"{cfg.seed}|{salt}|{x}".encode()).hexdigest())
    return q.sort_values("_smoke_key").head(int(n)).drop(columns="_smoke_key").reset_index(drop=True)

def smoke_split_take(train_df,val_df,total,salt):
    """Deterministically cap two already-disjoint splits without changing split membership."""
    if not smoke_enabled(): return train_df.reset_index(drop=True), val_df.reset_index(drop=True)
    total=min(int(total),len(train_df)+len(val_df))
    n_val=min(len(val_df), max(1, round(total*cfg.nih_val_fraction)))
    n_train=min(len(train_df), total-n_val)
    short=total-(n_train+n_val)
    if short>0:
        add=min(short,len(train_df)-n_train); n_train+=add; short-=add
    if short>0:
        n_val+=min(short,len(val_df)-n_val)
    a=stable_take(train_df,n_train,salt+"|train")
    b=stable_take(val_df,n_val,salt+"|val")
    log_event("Smoke split cap applied",level="OK",step="1.2",requested=total,train=len(a),validation=len(b))
    return a,b

log_event("Smoke/full execution controls initialized",level="OK",step="1.2",mode=cfg.run_mode,
          output=str(OUT),checkpoint_isolation=smoke_enabled())
cell_end(_CELL_T0,"1.2","Smoke-test controls and output isolation")


[+     0.0s] [CELL   ] [1.2] BEGIN CELL — Smoke-test controls and output isolation
[+     0.1s] [OK     ] [1.2] Smoke/full execution controls initialized | mode=full | output=/kaggle/working/qfid_matched_stage1_nih_only | checkpoint_isolation=False
[+     0.1s] [CELL   ] [1.2] END CELL — Smoke-test controls and output isolation | elapsed=0.00s


### 1.3 Runtime contract checks

In [3]:
_CELL_T0=cell_begin("1.3","Runtime contract checks")
def validate_runtime_contract():
    assert CLASSES == ("Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion")
    assert cfg.unfreeze_blocks == 2
    assert cfg.expected_cohort_size == 20746
    assert effective_nih_epochs() >= 1
    assert 0 < cfg.nih_val_fraction < 1
    log_event("NIH-only runtime contract passed",level="OK",step="1.3",mode=cfg.run_mode,
              expected_images=cfg.expected_cohort_size,chexpert_preadaptation=False,
              frozen_chexzero_reference=True,output=str(OUT),pipeline_version=PIPELINE_VERSION)
validate_runtime_contract()
cell_end(_CELL_T0,"1.3","Runtime contract checks")

[+     0.1s] [CELL   ] [1.3] BEGIN CELL — Runtime contract checks
[+     0.1s] [OK     ] [1.3] NIH-only runtime contract passed | mode=full | expected_images=20746 | chexpert_preadaptation=False | frozen_chexzero_reference=True | output=/kaggle/working/qfid_matched_stage1_nih_only | pipeline_version=stage1-nih-exact-one-complete-packs-no-chexpert-v12
[+     0.1s] [CELL   ] [1.3] END CELL — Runtime contract checks | elapsed=0.00s


### 1.4 Execution logging contract

Every executable cell emits a numbered **BEGIN/END** log. Long operations additionally emit **STEP**, **PROGRESS**, **TRAIN**, **EVAL**, **SAVE**, **LOAD**, **BEST**, **WARN**, **ERROR**, and **SUCCESS** records with elapsed time and relevant counts. Smoke mode logs every training/evaluation batch; full mode logs at a configurable interval. The same stream is appended to `execution_trace.log` and exported as `execution_events.json`.


## 2. Reproducibility, hashing, and atomic I/O
### 2.1 Global RNG control


In [4]:
_CELL_T0=cell_begin("2.1","Global RNG control")
def seed_all(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_all(cfg.seed)
cell_end(_CELL_T0,"2.1","Global RNG control")


[+     0.1s] [CELL   ] [2.1] BEGIN CELL — Global RNG control
[+     0.2s] [CELL   ] [2.1] END CELL — Global RNG control | elapsed=0.00s


### 2.2 Content hashing and atomic persistence


In [5]:
_CELL_T0=cell_begin("2.2","Content hashing and atomic persistence")
def sha256_file(path,chunk=8<<20):
    path=Path(path); t0=time.time()
    log_event("Hashing file",level="STEP",step="2.2",file=str(path),size_mb=round(path.stat().st_size/(1024**2),1) if path.exists() else "missing")
    try:
        h=hashlib.sha256()
        with open(path,"rb") as f:
            while True:
                b=f.read(chunk)
                if not b: break
                h.update(b)
        digest=h.hexdigest()
    except Exception as e:
        log_event("File hashing failed",level="ERROR",step="2.2",file=str(path),error=f"{type(e).__name__}: {e}")
        raise
    log_event("File hash complete",level="OK",step="2.2",file=path.name,sha256=digest[:16],elapsed=f"{time.time()-t0:.2f}s")
    return digest


def canonical_hash(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str,separators=(",",":")).encode()).hexdigest()


def atomic_json(path,obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+".tmp"); t0=time.time()
    log_event("Saving JSON",level="SAVE",step="2.2",file=str(path))
    try:
        tmp.write_text(json.dumps(obj,indent=2,default=str),encoding="utf-8")
        os.replace(tmp,path)
    except Exception as e:
        try:
            if tmp.exists(): tmp.unlink()
        except Exception: pass
        log_event("JSON save failed",level="ERROR",step="2.2",file=str(path),error=f"{type(e).__name__}: {e}")
        raise
    log_event("Saved JSON",level="OK",step="2.2",file=path.name,size_kb=round(path.stat().st_size/1024,1),elapsed=f"{time.time()-t0:.2f}s")


def atomic_torch(path,obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+".tmp"); t0=time.time()
    log_event("Saving torch checkpoint",level="SAVE",step="2.2",file=str(path))
    try:
        torch.save(obj,tmp)
        os.replace(tmp,path)
    except Exception as e:
        try:
            if tmp.exists(): tmp.unlink()
        except Exception: pass
        log_event("Torch checkpoint save failed",level="ERROR",step="2.2",file=str(path),error=f"{type(e).__name__}: {e}")
        raise
    log_event("Saved torch checkpoint",level="OK",step="2.2",file=path.name,size_mb=round(path.stat().st_size/(1024**2),1),elapsed=f"{time.time()-t0:.2f}s")

cell_end(_CELL_T0,"2.2","Content hashing and atomic persistence")


[+     0.2s] [CELL   ] [2.2] BEGIN CELL — Content hashing and atomic persistence
[+     0.2s] [CELL   ] [2.2] END CELL — Content hashing and atomic persistence | elapsed=0.00s


### 2.3 RNG restoration and Kaggle runtime guard


In [6]:
_CELL_T0=cell_begin("2.3","RNG restoration and Kaggle runtime guard")
def rng_state():
    return {"python":random.getstate(),"numpy":np.random.get_state(),"torch":torch.get_rng_state(),
            "cuda":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(s):
    if not s: return
    random.setstate(s["python"]); np.random.set_state(s["numpy"]); torch.set_rng_state(s["torch"])
    if s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])
    log_event("RNG state restored from checkpoint",level="OK",step="2.3")

def seconds_left():
    return cfg.runtime_limit_hours*3600-(time.time()-START_TIME)

def should_stop_before_next_unit(last_unit_seconds=None):
    guard=cfg.runtime_guard_minutes*60
    need=guard+(last_unit_seconds or 0)
    stop=seconds_left()<need
    if stop:
        log_event("Runtime guard requests graceful stop before next unit",level="WARN",step="2.3",
                  seconds_left=round(seconds_left(),1),estimated_need=round(need,1))
    return stop


def save_status(status,next=None,**extra):
    payload={
        "pipeline_version":PIPELINE_VERSION,
        "status":status,
        "next":next,
        "run_mode":cfg.run_mode,
        "smoke_test":smoke_enabled(),
        "elapsed_seconds":round(time.time()-START_TIME,3),
        "seconds_left":round(seconds_left(),3),
        "updated_unix":time.time(),
        **extra,
    }
    atomic_json(OUT/"run_status.json",payload)
    log_event("Run status updated",level="STATUS",step="2.3",status=status,next=next,**extra)
    return payload

cell_end(_CELL_T0,"2.3","RNG restoration and Kaggle runtime guard")


[+     0.2s] [CELL   ] [2.3] BEGIN CELL — RNG restoration and Kaggle runtime guard
[+     0.2s] [CELL   ] [2.3] END CELL — RNG restoration and Kaggle runtime guard | elapsed=0.00s


## 3. NIH mined-patch discovery
### 3.1 Metadata normalization and patch-record schema


In [7]:
_CELL_T0=cell_begin("3.1","Metadata normalization and patch-record schema")
def mdict(x):
    if isinstance(x,dict): return x
    if hasattr(x,"_asdict"): return x._asdict()
    if hasattr(x,"__dict__"): return vars(x)
    return {}

def pick(m,keys):
    for k in keys:
        if k in m and m[k] is not None: return m[k]
    return None

def infer_image_id(m,i=0):
    p=pick(m,("image_name","img_name","image_id","image_path","img_path","path","filename","file_name"))
    return Path(str(p)).name if p else f"unknown_{i}"

def infer_patient(m):
    p=pick(m,("patient_id","patient","subject_id"))
    return str(p) if p is not None else None

def infer_split(m):
    v=pick(m,("split","fold","subset","partition"))
    if v is None: return None
    x=str(v).strip().lower(); return "validation" if x in {"val","valid","validation"} else x

def bbox_from_meta(m):
    b=pick(m,("bbox","box","bounding_box","xyxy"))
    if b is not None:
        try:
            a=np.asarray(b,dtype=float).reshape(-1)
            if len(a)>=4: return tuple(float(x) for x in a[:4])
        except Exception: pass
    if all(k in m for k in ("x1","y1","x2","y2")):
        return tuple(float(m[k]) for k in ("x1","y1","x2","y2"))
    return None

def infer_image_path(m):
    p=pick(m,("image_path","img_path","path")); return str(p) if p else None

@dataclass
class PatchRec:
    image_id:str
    patient_id:str|None
    source_split:str|None
    bbox:tuple|None
    image_path:str|None
    bucket:int
    meta:dict
cell_end(_CELL_T0,"3.1","Metadata normalization and patch-record schema")


[+     0.3s] [CELL   ] [3.1] BEGIN CELL — Metadata normalization and patch-record schema
[+     0.3s] [CELL   ] [3.1] END CELL — Metadata normalization and patch-record schema | elapsed=0.00s


### 3.2 Stage-3 pack loading and file discovery


In [8]:
_CELL_T0=cell_begin("3.2","Stage-3 pack loading and file discovery")
def _extract_meta(obj):
    if isinstance(obj,(list,tuple)): return list(obj)
    if isinstance(obj,dict):
        for k in ("meta","metadata","records","rows"):
            if isinstance(obj.get(k),(list,tuple)): return list(obj[k])
    return None

def load_stage_pack(path,bucket):
    path=Path(path); t0=time.time()
    log_event("Loading NIH mined-patch pack",level="STEP",step="3.2",bucket=bucket,file=str(path))
    if not path.exists(): raise FileNotFoundError(path)
    try: obj=torch.load(path,map_location="cpu",weights_only=False)
    except TypeError: obj=torch.load(path,map_location="cpu")
    M=_extract_meta(obj)
    if M is None: raise RuntimeError(f"{path.name}: no metadata list found")
    bid={"causal":0,"spur_in":1,"spur_out":2}[bucket]
    out=[]
    for i,m0 in enumerate(M):
        m=mdict(m0).copy()
        out.append(PatchRec(infer_image_id(m,i),infer_patient(m),infer_split(m),bbox_from_meta(m),infer_image_path(m),bid,m))
    log_event("Loaded NIH mined-patch pack",level="OK",step="3.2",bucket=bucket,records=len(out),elapsed=f"{time.time()-t0:.2f}s")
    return out
cell_end(_CELL_T0,"3.2","Stage-3 pack loading and file discovery")


[+     0.3s] [CELL   ] [3.2] BEGIN CELL — Stage-3 pack loading and file discovery
[+     0.3s] [CELL   ] [3.2] END CELL — Stage-3 pack loading and file discovery | elapsed=0.00s


### 3.3 Original NIH radiograph resolver


In [9]:
_CELL_T0=cell_begin("3.3","Original NIH radiograph resolver")
def stage_paths():
    r=Path(cfg.nih_stage_root)
    paths={"causal":r/"stage3_causal_complete.pt",
           "spur_in":r/"stage3_spur_in_complete.pt",
           "spur_out":r/"stage3_spur_out_complete.pt"}
    for bucket,p in paths.items():
        if not p.exists(): raise FileNotFoundError(f"Missing required complete checkpoint: {p}")
        if not p.name.endswith("_complete.pt") or "_remainder" in p.name:
            raise RuntimeError(f"Refusing non-complete checkpoint for {bucket}: {p.name}")
    log_event("Resolved exactly three complete Stage-3 pack paths",level="OK",step="3.3",**{k:str(v) for k,v in paths.items()})
    return paths

def _nih_direct_search_dirs(root):
    root=Path(root); dirs=[]
    if not root.exists(): return dirs
    bases=[root,root/"data",root/"Data"]
    for base in bases:
        if not base.exists(): continue
        dirs.extend([base,base/"images"])
        try:
            for d in base.iterdir():
                if not d.is_dir(): continue
                name=d.name.lower()
                if name.startswith("images_") or name in {"images","chestx-ray14","chestxray14"}: dirs.extend([d,d/"images"])
        except Exception: pass
    seen=set(); out=[]
    for d in dirs:
        key=str(d)
        if key not in seen and d.exists(): seen.add(key); out.append(d)
    return out

def resolve_nih_images(records):
    t_all=time.time(); wanted={}
    for r in records:
        candidates=[]
        for v in [r.image_id,r.image_path,*[r.meta.get(k) for k in ("image_name","img_name","image_id","filename","file_name","path")]]:
            if v:
                b=Path(str(v)).name; candidates += [b,Path(b).stem]
        wanted[id(r)]=list(dict.fromkeys(candidates))
    unresolved=[r for r in records if not (r.image_path and Path(r.image_path).exists())]
    if not unresolved: return records
    roots=[]; conf=Path(cfg.nih_image_root)
    if conf.exists(): roots.append(conf)
    kin=Path("/kaggle/input"); stage_root=Path(cfg.nih_stage_root)
    if kin.exists():
        for p in kin.iterdir():
            try: same_stage=p.resolve()==stage_root.resolve()
            except Exception: same_stage=False
            if p.is_dir() and not same_stage and any(t in p.name.lower() for t in ("nih","chest","xray","x-ray")): roots.append(p)
    roots=list(dict.fromkeys(roots)); direct_dirs=[]
    for root in roots: direct_dirs.extend(_nih_direct_search_dirs(root))
    direct_dirs=list(dict.fromkeys(direct_dirs))
    for ri,r in enumerate(list(unresolved),1):
        names=[]
        for n in wanted[id(r)]:
            pp=Path(n); names.extend([pp.name] if pp.suffix else [pp.name+ext for ext in (".png",".jpg",".jpeg")])
        found=None
        for d in direct_dirs:
            for name in names:
                q=d/name
                if q.exists() and q.is_file(): found=str(q); break
            if found: break
        if found: r.image_path=found
        if ri%max(1,image_log_every())==0 or ri==len(unresolved): progress_log("3.3","NIH targeted path probes",ri,len(unresolved))
    unresolved=[r for r in records if not (r.image_path and Path(r.image_path).exists())]
    if unresolved:
        target_names=set(x for r in unresolved for x in wanted[id(r)]); lookup={}
        for root in roots:
            for dp,_,fns in os.walk(root):
                for fn in fns:
                    if fn in target_names or Path(fn).stem in target_names:
                        full=str(Path(dp)/fn); lookup.setdefault(fn,full); lookup.setdefault(Path(fn).stem,full)
            for r in unresolved:
                for n in wanted[id(r)]:
                    if n in lookup: r.image_path=lookup[n]; break
            unresolved=[r for r in unresolved if not (r.image_path and Path(r.image_path).exists())]
            if not unresolved: break
    if unresolved: raise RuntimeError(f"Could not resolve {len(unresolved)} original NIH radiographs. Examples: {[r.image_id for r in unresolved[:8]]}")
    log_event("All requested NIH radiographs resolved",level="OK",step="3.3",records=len(records),elapsed=f"{time.time()-t_all:.2f}s")
    return records
cell_end(_CELL_T0,"3.3","Original NIH radiograph resolver")

[+     0.4s] [CELL   ] [3.3] BEGIN CELL — Original NIH radiograph resolver
[+     0.4s] [CELL   ] [3.3] END CELL — Original NIH radiograph resolver | elapsed=0.00s


## 4. Matched NIH cohort construction
### 4.1 Official ChestX-ray14 label discovery and five-class mapping


In [10]:
_CELL_T0=cell_begin("4.1","Official ChestX-ray14 label discovery and five-class mapping")
def discover_nih_label_csv():
    log_event("Searching for official NIH ChestX-ray14 label CSV",level="STEP",step="4.1")
    if cfg.nih_official_csv:
        p=Path(cfg.nih_official_csv)
        if not p.exists(): raise FileNotFoundError(p)
        log_event("Using configured NIH label CSV",level="OK",step="4.1",file=str(p))
        return p
    candidates=[]
    # Search the configured NIH root first; global Kaggle search is only a fallback.
    preferred=[Path(cfg.nih_image_root)]
    for root in preferred:
        if root.exists():
            for name in ("Data_Entry_2017.csv","Data_Entry_2017_v2020.csv","nih_labels.csv"):
                candidates += list(root.rglob(name))
    if not candidates and Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").iterdir():
            if p.is_dir() and any(t in p.name.lower() for t in ("nih","chest","xray","x-ray")):
                for name in ("Data_Entry_2017.csv","Data_Entry_2017_v2020.csv","nih_labels.csv"):
                    candidates += list(p.rglob(name))
    if not candidates and Path("/kaggle/input").exists():
        log_event("Named NIH CSV not found; checking CSV headers as final fallback",level="WARN",step="4.1")
        for p in Path("/kaggle/input").rglob("*.csv"):
            try:
                h=pd.read_csv(p,nrows=2)
                if {"Image Index","Finding Labels"}.issubset(h.columns): candidates.append(p); break
            except Exception:
                pass
    if not candidates: raise FileNotFoundError("Official NIH ChestX-ray14 label CSV not found. Set cfg.nih_official_csv.")
    chosen=candidates[0]
    log_event("Official NIH label CSV found",level="OK",step="4.1",file=str(chosen),candidates=len(candidates))
    return chosen

def official_nih_labels(csv_path):
    t0=time.time(); log_event("Loading official NIH image-level labels",level="STEP",step="4.1",file=str(csv_path))
    df=pd.read_csv(csv_path)
    if not {"Image Index","Finding Labels"}.issubset(df.columns):
        raise RuntimeError("NIH label CSV must contain 'Image Index' and 'Finding Labels'.")
    out={}; positives=Counter()
    for _,r in df.iterrows():
        image=Path(str(r["Image Index"])).name
        findings={x.strip() for x in str(r["Finding Labels"]).split("|")}
        aliases={"Pleural Effusion":"Effusion"}
        vec=np.asarray([1.0 if aliases.get(c,c) in findings else 0.0 for c in CLASSES],dtype=np.float32)
        for j,c in enumerate(CLASSES): positives[c]+=int(vec[j])
        out[image]=vec; out[Path(image).stem]=vec
    log_event("Official NIH labels loaded",level="OK",step="4.1",rows=len(df),lookup_keys=len(out),
              positives=dict(positives),elapsed=f"{time.time()-t0:.2f}s")
    return out
cell_end(_CELL_T0,"4.1","Official ChestX-ray14 label discovery and five-class mapping")


[+     0.4s] [CELL   ] [4.1] BEGIN CELL — Official ChestX-ray14 label discovery and five-class mapping
[+     0.4s] [CELL   ] [4.1] END CELL — Official ChestX-ray14 label discovery and five-class mapping | elapsed=0.00s


### 4.2 Official exactly-one-of-five cohort verification

In [11]:
_CELL_T0=cell_begin("4.2","Official exactly-one-of-five cohort verification")
def load_exact_one_cohort():
    p=Path(cfg.exact_one_csv)
    if not p.exists(): raise FileNotFoundError(f"Exactly-one-of-five cohort CSV not found: {p}")
    df=pd.read_csv(p)
    image_col=next((c for c in ("Image Index","image_id","image_name","filename","file_name","image") if c in df.columns),None)
    if image_col is None: image_col=next((c for c in df.columns if "image" in c.lower() and "path" not in c.lower()),None)
    if image_col is None: raise RuntimeError(f"No image identifier column in {p.name}: {list(df.columns)}")
    df["image_id"]=df[image_col].astype(str).map(lambda x:Path(x).name)
    if all(c in df.columns for c in LABEL_COLS): y=df[list(LABEL_COLS)].apply(pd.to_numeric,errors="coerce").to_numpy(float)
    elif all(c in df.columns for c in CLASSES): y=df[list(CLASSES)].apply(pd.to_numeric,errors="coerce").to_numpy(float)
    elif "Finding Labels" in df.columns:
        aliases={"Pleural Effusion":"Effusion"}; y=[]
        for text in df["Finding Labels"].astype(str):
            findings={x.strip() for x in text.split("|")}; y.append([float(aliases.get(c,c) in findings) for c in CLASSES])
        y=np.asarray(y,float)
    else:
        official=official_nih_labels(discover_nih_label_csv()); y=[]
        for image_id in df.image_id:
            v=official.get(image_id,official.get(Path(image_id).stem))
            if v is None: raise RuntimeError(f"No official NIH label for {image_id}")
            y.append(v)
        y=np.asarray(y,float)
    for j,c in enumerate(LABEL_COLS): df[c]=y[:,j]
    df=df[["image_id",*LABEL_COLS]].drop_duplicates("image_id").reset_index(drop=True)
    if not np.isfinite(df[list(LABEL_COLS)].to_numpy(float)).all(): raise RuntimeError("Non-finite cohort labels.")
    if not df[list(LABEL_COLS)].sum(axis=1).eq(1).all(): raise RuntimeError("Cohort is not exactly one-positive-of-five.")
    if len(df)!=cfg.expected_cohort_size: raise RuntimeError(f"Cohort has {len(df):,} images; expected {cfg.expected_cohort_size:,}.")
    log_event("Official exactly-one-of-five cohort verified",level="OK",step="4.2",images=len(df),
              class_counts={CLASSES[j]:int(df[LABEL_COLS[j]].sum()) for j in range(N_CLASSES)})
    return df
exact_cohort=load_exact_one_cohort()
cell_end(_CELL_T0,"4.2","Official exactly-one-of-five cohort verification")

[+     0.5s] [CELL   ] [4.2] BEGIN CELL — Official exactly-one-of-five cohort verification
[+     0.8s] [OK     ] [4.2] Official exactly-one-of-five cohort verified | images=20746 | class_counts={'Atelectasis': 7320, 'Cardiomegaly': 1436, 'Consolidation': 2516, 'Edema': 1436, 'Pleural Effusion': 8038}
[+     0.8s] [CELL   ] [4.2] END CELL — Official exactly-one-of-five cohort verification | elapsed=0.28s


### 4.2 Patient identity, image decoding, bounding-box validation, and split rule


In [12]:
_CELL_T0=cell_begin("4.2","Patient identity, image decoding, bounding-box validation, and split rule")
def patient_from_nih_image(image_id):
    m=re.match(r"(\d{8})_\d+",Path(image_id).stem)
    return m.group(1) if m else None

def valid_bbox(path,bbox):
    if bbox is None: return False
    a=np.asarray(bbox,float)
    if a.shape!=(4,) or not np.isfinite(a).all(): return False
    with Image.open(path) as im: w,h=im.size
    x1,y1,x2,y2=a.tolist()
    if max(abs(x1),abs(y1),abs(x2),abs(y2))<=1.5:
        x1,x2=x1*w,x2*w; y1,y2=y1*h,y2*h
    return 0<=x1<x2<=w and 0<=y1<y2<=h

def decode_ok(path):
    try:
        with Image.open(path) as im: im.verify()
        return True
    except Exception: return False

def patient_partition(patient_id,seed=42,val_fraction=.2):
    u=int(hashlib.sha256(f"{seed}|nih-patient|{patient_id}".encode()).hexdigest()[:16],16)/(16**16)
    return "validation" if u<val_fraction else "train"
cell_end(_CELL_T0,"4.2","Patient identity, image decoding, bounding-box validation, and split rule")


[+     0.8s] [CELL   ] [4.2] BEGIN CELL — Patient identity, image decoding, bounding-box validation, and split rule
[+     0.8s] [CELL   ] [4.2] END CELL — Patient identity, image decoding, bounding-box validation, and split rule | elapsed=0.00s


In [13]:
_CELL_T0=cell_begin("4.3","Construct, validate, audit, and save the matched cohort")
def _early_smoke_nih_records(recs):
    """Shrink RAW patch metadata before any expensive NIH filesystem walk or PIL validation.

    We keep all mined patches for each selected image, but select only a small deterministic
    image pool (default 4 x target) so invalid/missing candidates can be dropped while still
    leaving enough rows for the final 50-image smoke manifest.
    """
    if not smoke_enabled():
        log_event("NIH early cap skipped in FULL mode",level="INFO",step="4.3",patch_records=len(recs))
        return recs
    groups=defaultdict(list)
    for r in recs:
        groups[Path(r.image_id).name].append(r)
    image_ids=sorted(groups)
    ranked=sorted(image_ids,key=lambda x:hashlib.sha256(f"{cfg.seed}|nih-early-smoke|{x}".encode()).hexdigest())
    n=min(len(ranked), max(cfg.smoke_nih_images, cfg.smoke_nih_images*cfg.smoke_candidate_multiplier))
    chosen=set(ranked[:n])
    out=[r for image_id in ranked[:n] for r in groups[image_id]]
    log_event("Early NIH smoke cap applied BEFORE filesystem resolution",level="OK",step="4.3",
              raw_patches=len(recs),raw_images=len(groups),candidate_patches=len(out),candidate_images=len(chosen))
    return out


def construct_matched_manifest():
    t_all=time.time(); log_event("Constructing matched NIH cohort",level="STEP",step="4.3")
    packs=stage_paths(); recs=[]
    for k in ("causal","spur_in","spur_out"):
        part=load_stage_pack(packs[k],k); recs += part
        log_event("Accumulated NIH patch records",level="INFO",step="4.3",bucket=k,bucket_records=len(part),running_total=len(recs))

    exact_labels={r.image_id:r[list(LABEL_COLS)].to_numpy(dtype=np.float32) for _,r in exact_cohort.iterrows()}
    allowed=set(exact_labels)
    recs=[r for r in recs if Path(r.image_id).name in allowed]
    recs=_early_smoke_nih_records(recs)
    recs=resolve_nih_images(recs)
    labels=exact_labels

    grouped=defaultdict(list)
    for r in recs: grouped[Path(r.image_id).name].append(r)
    log_event("Grouped complete-pack patches by official NIH image",level="OK",step="4.3",candidate_images=len(grouped),candidate_patches=len(recs))
    if not smoke_enabled() and len(grouped)!=cfg.expected_cohort_size:
        missing=sorted(set(exact_labels)-set(grouped))
        raise RuntimeError(f"Complete packs cover {len(grouped):,}/{cfg.expected_cohort_size:,} official images; examples missing: {missing[:10]}")

    rows=[]; audit=Counter(); every=max(1,image_log_every())
    for ix,(image_id,grp) in enumerate(sorted(grouped.items()),1):
        path=next((g.image_path for g in grp if g.image_path and Path(g.image_path).exists()),None)
        if not path or not decode_ok(path): audit["missing_or_bad_whole"]+=1; continue
        labs=labels.get(Path(path).name)
        if labs is None: labs=labels.get(Path(image_id).name)
        if labs is None: labs=labels.get(Path(image_id).stem)
        if labs is None: audit["missing_official_labels"]+=1; continue
        patient=next((g.patient_id for g in grp if g.patient_id),None) or patient_from_nih_image(image_id)
        if not patient: audit["missing_patient"]+=1; continue
        good=[]; seen=set()
        for g in grp:
            if not valid_bbox(path,g.bbox): audit["invalid_bbox"]+=1; continue
            b=tuple(float(x) for x in g.bbox); key=(tuple(round(x,5) for x in b),g.bucket)
            if key in seen: continue
            seen.add(key); good.append({"bbox":b,"bucket":int(g.bucket)})
        if not good: audit["no_valid_patch"]+=1; continue
        split=patient_partition(str(patient),cfg.seed,cfg.nih_val_fraction)
        rows.append({"patient_id":str(patient),"image_id":Path(image_id).name,"whole_image_path":str(path),
                     "bbox_json":json.dumps([x["bbox"] for x in good]),"bucket_json":json.dumps([x["bucket"] for x in good]),
                     **{LABEL_COLS[j]:float(labs[j]) for j in range(N_CLASSES)},"source_split":split})
        audit["retained_images"]+=1; audit["retained_patches"]+=len(good)
        if ix%every==0 or ix==len(grouped):
            progress_log("4.3","NIH image validation",ix,len(grouped),retained=len(rows),invalid_bbox=audit["invalid_bbox"])
    df=pd.DataFrame(rows)
    if df.empty: raise RuntimeError("Matched NIH cohort is empty.")

    if smoke_enabled():
        tr0=df[df.source_split.eq("train")].copy(); va0=df[df.source_split.eq("validation")].copy()
        tr0,va0=smoke_split_take(tr0,va0,cfg.smoke_nih_images,"nih")
        df=pd.concat([tr0,va0],ignore_index=True)
        if len(df)<cfg.smoke_nih_images:
            log_event("Fewer than the requested NIH smoke images survived the candidate pool",level="WARN",step="4.3",
                      retained=len(df),requested=cfg.smoke_nih_images,suggestion="Increase cfg.smoke_candidate_multiplier")
        log_event("Final NIH smoke cap",level="OK",step="4.3",total=len(df),train=len(tr0),validation=len(va0))

    if not df[list(LABEL_COLS)].sum(axis=1).eq(1).all(): raise RuntimeError("Final manifest is not exactly one-positive-of-five.")
    if not smoke_enabled() and len(df)!=cfg.expected_cohort_size:
        raise RuntimeError(f"Validated full cohort has {len(df):,}/{cfg.expected_cohort_size:,} images; audit={dict(audit)}")
    tr=set(df.loc[df.source_split.eq("train"),"patient_id"]); va=set(df.loc[df.source_split.eq("validation"),"patient_id"])
    if tr&va: raise RuntimeError(f"NIH patient leakage: {len(tr&va)}")
    df.to_csv(OUT/"matched_nih_manifest.csv",index=False)
    df[df.source_split.eq("train")].to_csv(OUT/"matched_nih_train.csv",index=False)
    df[df.source_split.eq("validation")].to_csv(OUT/"matched_nih_validation.csv",index=False)
    audit.update({"train_images":int((df.source_split=="train").sum()),"validation_images":int((df.source_split=="validation").sum()),
                  "train_patients":len(tr),"validation_patients":len(va)})
    atomic_json(OUT/"matched_nih_audit.json",dict(audit))
    log_event("Matched NIH cohort complete",level="OK",step="4.3",audit=dict(audit),elapsed=f"{time.time()-t_all:.2f}s")
    return df

with timed_step("4.3","End-to-end matched NIH manifest construction"):
    nih_manifest=construct_matched_manifest()
cell_end(_CELL_T0,"4.3","Construct, validate, audit, and save the matched cohort")


[+     0.9s] [CELL   ] [4.3] BEGIN CELL — Construct, validate, audit, and save the matched cohort
[+     0.9s] [STEP   ] [4.3] START — End-to-end matched NIH manifest construction
[+     0.9s] [STEP   ] [4.3] Constructing matched NIH cohort
[+     0.9s] [OK     ] [3.3] Resolved exactly three complete Stage-3 pack paths | causal=/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/stage3_causal_complete.pt | spur_in=/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/stage3_spur_in_complete.pt | spur_out=/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/stage3_spur_out_complete.pt
[+     0.9s] [STEP   ] [3.2] Loading NIH mined-patch pack | bucket=causal | file=/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/stage3_causal_complete.pt
[+     5.2s] [OK     ] [3.2] Loaded NIH mined-patch pack | bucket=causal | records=118658 | elapsed=4.32s
[+     5.2s] [INFO   ] [4.3] Accumulate

### 4.4 NIH leakage and manifest sanity checks


In [14]:
_CELL_T0=cell_begin("4.4","NIH leakage and manifest sanity checks")
def audit_nih_manifest(df):
    log_event("Auditing matched NIH manifest",level="STEP",step="4.4",rows=len(df))
    required={"patient_id","image_id","whole_image_path","bbox_json","bucket_json","source_split",*LABEL_COLS}
    missing=required-set(df.columns)
    if missing: raise RuntimeError(f"NIH manifest missing columns: {sorted(missing)}")
    if df.image_id.duplicated().any(): raise RuntimeError("Matched NIH manifest must contain one row per original image.")
    tr=set(df.loc[df.source_split.eq("train"),"patient_id"].astype(str))
    va=set(df.loc[df.source_split.eq("validation"),"patient_id"].astype(str))
    if tr & va: raise RuntimeError("NIH patient leakage detected after manifest construction.")
    if not set(df.source_split.unique()).issubset({"train","validation"}): raise RuntimeError("Unexpected NIH split label.")
    if not np.isfinite(df[list(LABEL_COLS)].to_numpy(float)).all(): raise RuntimeError("Official NIH targets must be fully observed 0/1 labels.")
    if not df[list(LABEL_COLS)].sum(axis=1).eq(1).all(): raise RuntimeError("Every NIH image must have exactly one positive target.")
    if not smoke_enabled() and len(df)!=cfg.expected_cohort_size: raise RuntimeError("Full cohort must contain exactly 20,746 images.")
    label_prevalence={CLASSES[j]:round(float(df[LABEL_COLS[j]].mean()),4) for j in range(N_CLASSES)}
    log_event("NIH manifest audit passed",level="OK",step="4.4",images=len(df),patients=df.patient_id.nunique(),
              train_patients=len(tr),validation_patients=len(va),label_prevalence=label_prevalence)

audit_nih_manifest(nih_manifest)
cell_end(_CELL_T0,"4.4","NIH leakage and manifest sanity checks")


[+  1161.5s] [CELL   ] [4.4] BEGIN CELL — NIH leakage and manifest sanity checks
[+  1161.5s] [STEP   ] [4.4] Auditing matched NIH manifest | rows=20746
[+  1161.5s] [OK     ] [4.4] NIH manifest audit passed | images=20746 | patients=7276 | train_patients=5802 | validation_patients=1474 | label_prevalence={'Atelectasis': 0.3528, 'Cardiomegaly': 0.0692, 'Consolidation': 0.1213, 'Edema': 0.0692, 'Pleural Effusion': 0.3874}
[+  1161.5s] [CELL   ] [4.4] END CELL — NIH leakage and manifest sanity checks | elapsed=0.02s


## 6. Image transforms and datasets
### 6.1 Train/evaluation transforms and crop utilities


In [15]:
_CELL_T0=cell_begin("6.1","Train/evaluation transforms and crop utilities")
from torchvision import transforms as T

TRAIN_T=T.Compose([T.Resize((224,224)),T.RandomAffine(degrees=3,translate=(.02,.02),scale=(.97,1.03)),
                   T.ToTensor(),T.Normalize((.485,.456,.406),(.229,.224,.225))])
EVAL_T=T.Compose([T.Resize((224,224)),T.ToTensor(),T.Normalize((.485,.456,.406),(.229,.224,.225))])

def load_rgb(path):
    with Image.open(path) as im:
        return im.convert("RGB")
def crop_box(im,b):
    x1,y1,x2,y2=[float(x) for x in b]; w,h=im.size
    if max(abs(x1),abs(y1),abs(x2),abs(y2))<=1.5: x1,x2=x1*w,x2*w; y1,y2=y1*h,y2*h
    x1=max(0,min(w-1,int(round(x1)))); x2=max(x1+1,min(w,int(round(x2))))
    y1=max(0,min(h-1,int(round(y1)))); y2=max(y1+1,min(h,int(round(y2))))
    return im.crop((x1,y1,x2,y2))

def target_candidate_crops(im,k=6):
    im=im.convert("RGB"); w,h=im.size
    boxes=[(.05,.05,.55,.95),(.45,.05,.95,.95),(.10,.02,.90,.58),(.10,.42,.90,.98),(.18,.18,.82,.82),(.28,.08,.72,.92)][:k]
    return [im.crop((int(a*w),int(b*h),int(c*w),int(d*h))) for a,b,c,d in boxes]
cell_end(_CELL_T0,"6.1","Train/evaluation transforms and crop utilities")


[+  1161.6s] [CELL   ] [6.1] BEGIN CELL — Train/evaluation transforms and crop utilities
[+  1161.6s] [CELL   ] [6.1] END CELL — Train/evaluation transforms and crop utilities | elapsed=0.00s


### 6.2 Matched NIH whole-image and patch-bag datasets


In [16]:
_CELL_T0=cell_begin("6.2","Matched NIH whole-image and patch-bag datasets")
class NIHWholeDS(Dataset):
    def __init__(self,df,train): self.df=df.reset_index(drop=True); self.t=TRAIN_T if train else EVAL_T
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; y=torch.tensor(r[list(LABEL_COLS)].to_numpy(dtype=np.float32))
        return self.t(load_rgb(r.whole_image_path)),y,r.image_id,r.patient_id

class NIHPatchBagDS(Dataset):
    def __init__(self,df,train): self.df=df.reset_index(drop=True); self.t=TRAIN_T if train else EVAL_T
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; im=load_rgb(r.whole_image_path); boxes=json.loads(r.bbox_json)
        xs=torch.stack([self.t(crop_box(im,b)) for b in boxes])
        y=torch.tensor(r[list(LABEL_COLS)].to_numpy(dtype=np.float32))
        return xs,y,r.image_id,r.patient_id
cell_end(_CELL_T0,"6.2","Matched NIH whole-image and patch-bag datasets")


[+  1161.7s] [CELL   ] [6.2] BEGIN CELL — Matched NIH whole-image and patch-bag datasets
[+  1161.7s] [CELL   ] [6.2] END CELL — Matched NIH whole-image and patch-bag datasets | elapsed=0.00s


### 6.3 Variable-size NIH patch-bag collation

In [17]:
_CELL_T0=cell_begin("6.3","Variable-size NIH patch-bag collation")
def collate_patch_bags(batch):
    xs=[]; owners=[]; ys=[]; ids=[]; pids=[]
    for i,(bag,y,img,pid) in enumerate(batch):
        xs.append(bag); owners.append(torch.full((len(bag),),i,dtype=torch.long)); ys.append(y); ids.append(img); pids.append(pid)
    return torch.cat(xs),torch.cat(owners),torch.stack(ys),ids,pids
cell_end(_CELL_T0,"6.3","Variable-size NIH patch-bag collation")

[+  1161.8s] [CELL   ] [6.3] BEGIN CELL — Variable-size NIH patch-bag collation
[+  1161.8s] [CELL   ] [6.3] END CELL — Variable-size NIH patch-bag collation | elapsed=0.00s


## 7. Two independent ImageNet ViT-B/16 branches
### 7.1 Disentanglement head and orthogonality objective


In [18]:
_CELL_T0=cell_begin("7.1","Disentanglement head and orthogonality objective")
from torchvision.models import vit_b_16, ViT_B_16_Weights

class DisentangleHead(nn.Module):
    def __init__(self,in_dim=768,shared=256,out=128,n_classes=5):
        super().__init__()
        self.shared=nn.Sequential(nn.LayerNorm(in_dim),nn.Linear(in_dim,shared),nn.GELU(),nn.Dropout(.1),nn.LayerNorm(shared))
        self.causal=nn.Sequential(nn.Linear(shared,out),nn.GELU(),nn.LayerNorm(out))
        self.spurious=nn.Sequential(nn.Linear(shared,out),nn.GELU(),nn.LayerNorm(out))
        self.path_head=nn.Linear(out,n_classes)
    def forward(self,h):
        s=self.shared(h); zc=F.normalize(self.causal(s),dim=-1); zs=F.normalize(self.spurious(s),dim=-1)
        return {"zc":zc,"zs":zs,"logits":self.path_head(zc)}

def orthogonality_loss(zc,zs):
    zc=zc-zc.mean(0,keepdim=True); zs=zs-zs.mean(0,keepdim=True)
    return ((zc.T@zs)/max(1,len(zc))).pow(2).mean()
cell_end(_CELL_T0,"7.1","Disentanglement head and orthogonality objective")


[+  1161.9s] [CELL   ] [7.1] BEGIN CELL — Disentanglement head and orthogonality objective
[+  1161.9s] [CELL   ] [7.1] END CELL — Disentanglement head and orthogonality objective | elapsed=0.00s


### 7.2 ViT-B/16 trainable-depth policy and state hashing


In [19]:
_CELL_T0=cell_begin("7.2","ViT-B/16 trainable-depth policy and state hashing")
class ImageNetViTBranch(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__()
        weights=ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
        self.encoder=vit_b_16(weights=weights)
        self.encoder.heads=nn.Identity()
        self.head=DisentangleHead()
        self.freeze_except_last2()
    def freeze_except_last2(self):
        for p in self.encoder.parameters(): p.requires_grad=False
        for block in list(self.encoder.encoder.layers)[-cfg.unfreeze_blocks:]:
            for p in block.parameters(): p.requires_grad=True
        for p in self.encoder.encoder.ln.parameters(): p.requires_grad=True
        for p in self.head.parameters(): p.requires_grad=True
    def forward(self,x): return self.head(self.encoder(x))

def state_hash(model):
    h=hashlib.sha256()
    for n,t in sorted(model.state_dict().items()):
        a=t.detach().cpu().contiguous(); h.update(n.encode()); h.update(a.numpy().tobytes())
    return h.hexdigest()
cell_end(_CELL_T0,"7.2","ViT-B/16 trainable-depth policy and state hashing")


[+  1162.0s] [CELL   ] [7.2] BEGIN CELL — ViT-B/16 trainable-depth policy and state hashing
[+  1162.1s] [CELL   ] [7.2] END CELL — ViT-B/16 trainable-depth policy and state hashing | elapsed=0.00s


### 7.3 Identical initialization → independent whole and patch models


In [20]:
_CELL_T0=cell_begin("7.3","Identical initialization → independent whole and patch models")
log_event("Loading ImageNet ViT-B/16 initialization once",level="STEP",step="7.3")
t0=time.time(); _base=ImageNetViTBranch(pretrained=True).to(DEVICE)
IMAGENET_VIT_HASH=state_hash(_base.encoder)
base_state={k:v.detach().cpu().clone() for k,v in _base.state_dict().items()}
log_event("Base ImageNet ViT loaded and hashed",level="OK",step="7.3",sha256=IMAGENET_VIT_HASH[:16],elapsed=f"{time.time()-t0:.2f}s")

t0=time.time(); whole=ImageNetViTBranch(pretrained=False).to(DEVICE); whole.load_state_dict(base_state)
log_event("Independent whole-image branch initialized",level="OK",step="7.3",elapsed=f"{time.time()-t0:.2f}s")
t0=time.time(); patch=ImageNetViTBranch(pretrained=False).to(DEVICE); patch.load_state_dict(base_state)
log_event("Independent patch branch initialized",level="OK",step="7.3",elapsed=f"{time.time()-t0:.2f}s")
del _base,base_state; torch.cuda.empty_cache()

assert state_hash(whole.encoder)==state_hash(patch.encoder)==IMAGENET_VIT_HASH
assert whole is not patch and whole.encoder is not patch.encoder
log_event("Independent branch initialization verified",level="OK",step="7.3",imagenet_hash=IMAGENET_VIT_HASH[:16],
          whole_trainable=sum(p.numel() for p in whole.parameters() if p.requires_grad),
          patch_trainable=sum(p.numel() for p in patch.parameters() if p.requires_grad),**gpu_snapshot())
cell_end(_CELL_T0,"7.3","Identical initialization → independent whole and patch models")


[+  1162.2s] [CELL   ] [7.3] BEGIN CELL — Identical initialization → independent whole and patch models
[+  1162.2s] [STEP   ] [7.3] Loading ImageNet ViT-B/16 initialization once
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 146MB/s]


[+  1167.3s] [OK     ] [7.3] Base ImageNet ViT loaded and hashed | sha256=e5967821a6a1a01f | elapsed=5.14s
[+  1168.3s] [OK     ] [7.3] Independent whole-image branch initialized | elapsed=1.04s
[+  1169.3s] [OK     ] [7.3] Independent patch branch initialized | elapsed=0.95s
[+  1171.4s] [OK     ] [7.3] Independent branch initialization verified | imagenet_hash=e5967821a6a1a01f | whole_trainable=14443141 | patch_trainable=14443141 | gpu0_alloc_mb=656.6 | gpu0_reserved_mb=788.0 | gpu1_alloc_mb=0.0 | gpu1_reserved_mb=0.0
[+  1171.4s] [CELL   ] [7.3] END CELL — Identical initialization → independent whole and patch models | elapsed=9.24s


### 7.4 Trainable-parameter contract audit


In [21]:
_CELL_T0=cell_begin("7.4","Trainable-parameter contract audit")
def audit_trainable_depth(model,name):
    trainable=[n for n,p in model.named_parameters() if p.requires_grad]
    forbidden=[]
    layers=list(model.encoder.encoder.layers)
    for n in trainable:
        if n.startswith("encoder.") and not (
            any(n.startswith(f"encoder.encoder.layers.encoder_layer_{i}.") for i in range(len(layers)-cfg.unfreeze_blocks,len(layers)))
            or n.startswith("encoder.encoder.ln.")
        ): forbidden.append(n)
    if forbidden: raise RuntimeError(f"{name}: unexpected trainable encoder parameters: {forbidden[:8]}")
    if not any(n.startswith("head.") for n in trainable): raise RuntimeError(f"{name}: head is not trainable")
    log_event("Trainable-depth audit passed",level="OK",step="7.4",branch=name,
              trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),trainable_tensors=len(trainable))

audit_trainable_depth(whole,"whole")
audit_trainable_depth(patch,"patch")
cell_end(_CELL_T0,"7.4","Trainable-parameter contract audit")


[+  1171.5s] [CELL   ] [7.4] BEGIN CELL — Trainable-parameter contract audit
[+  1171.5s] [OK     ] [7.4] Trainable-depth audit passed | branch=whole | trainable_parameters=14443141 | trainable_tensors=42
[+  1171.5s] [OK     ] [7.4] Trainable-depth audit passed | branch=patch | trainable_parameters=14443141 | trainable_tensors=42
[+  1171.5s] [CELL   ] [7.4] END CELL — Trainable-parameter contract audit | elapsed=0.00s


## 8. Frozen CheXzero ViT-B/32 reference
### 8.1 Pathology prompts and whole-image dataset


In [22]:
_CELL_T0=cell_begin("8.1","Pathology prompts and whole-image dataset")
CHEXZERO_PHRASES={
"Atelectasis":["atelectasis","atelectatic changes","lung collapse"],
"Cardiomegaly":["cardiomegaly","enlarged cardiac silhouette","an enlarged heart"],
"Consolidation":["consolidation","airspace consolidation","lung consolidation"],
"Edema":["pulmonary edema","edema","interstitial pulmonary edema"],
"Pleural Effusion":["pleural effusion","a pleural effusion","fluid in the pleural space"],}
POS_TEMPLATES=["{}","there is {}","findings consistent with {}"]
NEG_TEMPLATES=["no {}","without {}","no evidence of {}"]

class PathDS(Dataset):
    def __init__(self,paths,t): self.paths=list(paths); self.t=t
    def __len__(self): return len(self.paths)
    def __getitem__(self,i): return self.t(load_rgb(self.paths[i]))
cell_end(_CELL_T0,"8.1","Pathology prompts and whole-image dataset")


[+  1171.6s] [CELL   ] [8.1] BEGIN CELL — Pathology prompts and whole-image dataset
[+  1171.6s] [CELL   ] [8.1] END CELL — Pathology prompts and whole-image dataset | elapsed=0.00s


### 8.2 Frozen model loading, checkpoint-coverage validation, and text anchors


In [23]:
_CELL_T0=cell_begin("8.2","Frozen model loading, checkpoint-coverage validation, and text anchors")
class FrozenCheXzero:
    def __init__(self,checkpoint):
        t0=time.time(); log_event("Initializing frozen CheXzero ViT-B/32",level="STEP",step="8.2",checkpoint=str(checkpoint))
        try: import open_clip
        except Exception as e: raise RuntimeError("open_clip_torch dependency preflight failed. Enable Kaggle Internet and rerun from the first cell.") from e
        self.open_clip=open_clip; self.checkpoint=str(checkpoint); self.checkpoint_hash=sha256_file(checkpoint)
        log_event("Creating CheXzero ViT-B/32 architecture",level="STEP",step="8.2")
        model,_,preprocess=open_clip.create_model_and_transforms("ViT-B-32",pretrained=None)
        log_event("Loading CheXzero checkpoint tensors",level="STEP",step="8.2")
        ck=torch.load(checkpoint,map_location=DEVICE,weights_only=False); sd=ck.get("state_dict",ck) if isinstance(ck,dict) else ck
        cleaned={}
        for k,v in sd.items():
            nk=k
            for pre in ("module.","model."):
                if nk.startswith(pre): nk=nk[len(pre):]
            cleaned[nk]=v
        own=model.state_dict(); matched={k:v for k,v in cleaned.items() if k in own and own[k].shape==v.shape}
        cov=sum(v.numel() for v in matched.values())/sum(v.numel() for v in own.values())
        log_event("CheXzero checkpoint coverage calculated",level="INFO",step="8.2",coverage=f"{cov:.2%}",matched_tensors=len(matched),model_tensors=len(own))
        if cov<.95: raise RuntimeError(f"CheXzero checkpoint coverage too low: {cov:.2%}")
        own.update(matched); model.load_state_dict(own,strict=True)
        self.model=model.to(DEVICE).eval(); self.preprocess=preprocess; self.coverage=cov
        for p in self.model.parameters(): p.requires_grad=False
        self._build_text()
        log_event("Frozen CheXzero initialized",level="OK",step="8.2",coverage=f"{cov:.2%}",elapsed=f"{time.time()-t0:.2f}s",**gpu_snapshot())
    @torch.no_grad()
    def _build_text(self):
        tok=self.open_clip.get_tokenizer("ViT-B-32")
        pos=[]; neg=[]
        log_event("Building frozen CheXzero pathology text anchors",level="STEP",step="8.2",classes=N_CLASSES)
        for idx,c in enumerate(CLASSES,1):
            p=[t.format(x) for x in CHEXZERO_PHRASES[c] for t in POS_TEMPLATES]
            n=[t.format(x) for x in CHEXZERO_PHRASES[c] for t in NEG_TEMPLATES]
            pe=F.normalize(self.model.encode_text(tok(p).to(DEVICE)).float(),dim=1).mean(0)
            ne=F.normalize(self.model.encode_text(tok(n).to(DEVICE)).float(),dim=1).mean(0)
            pos.append(F.normalize(pe,dim=0)); neg.append(F.normalize(ne,dim=0))
            progress_log("8.2","CheXzero text-anchor encoding",idx,N_CLASSES,pathology=c)
        self.pos=torch.stack(pos); self.neg=torch.stack(neg)
    @torch.no_grad()
    def encode_paths(self,paths,batch=64):
        workers=effective_num_workers()
        dl=DataLoader(PathDS(paths,self.preprocess),batch_size=batch,shuffle=False,num_workers=workers,pin_memory=True)
        z=[]; total=len(dl); t0=time.time()
        log_event("Encoding frozen CheXzero whole-image features",level="STEP",step="13.1",images=len(paths),batches=total,batch_size=batch,workers=workers)
        for bi,x in enumerate(dl,1):
            z.append(F.normalize(self.model.encode_image(x.to(DEVICE,non_blocking=True)).float(),dim=1).cpu())
            if bi%max(1,batch_log_every())==0 or bi==total:
                progress_log("13.1","CheXzero image encoding",bi,total,encoded=min(bi*batch,len(paths)),**gpu_snapshot())
        out=torch.cat(z) if z else torch.empty(0,512)
        log_event("CheXzero image encoding complete",level="OK",step="13.1",shape=tuple(out.shape),elapsed=f"{time.time()-t0:.2f}s")
        return out
    def margins(self,z):
        z=F.normalize(z.to(DEVICE).float(),dim=1); return ((z@self.pos.T)-(z@self.neg.T)).cpu()
    def positive_similarity(self,z):
        z=F.normalize(z.to(DEVICE).float(),dim=1); return (z@self.pos.T).cpu()
cell_end(_CELL_T0,"8.2","Frozen model loading, checkpoint-coverage validation, and text anchors")


[+  1171.7s] [CELL   ] [8.2] BEGIN CELL — Frozen model loading, checkpoint-coverage validation, and text anchors
[+  1171.7s] [CELL   ] [8.2] END CELL — Frozen model loading, checkpoint-coverage validation, and text anchors | elapsed=0.00s


### 8.3 Instantiate and persist the immutable CheXzero reference


In [24]:
_CELL_T0=cell_begin("8.3","Instantiate and persist the immutable CheXzero reference")
with timed_step("8.3","Instantiate and persist immutable CheXzero reference"):
    chexzero=FrozenCheXzero(cfg.chexzero_checkpoint)
    chexzero_ref={"architecture":"medical CLIP ViT-B/32","checkpoint":cfg.chexzero_checkpoint,
                  "checkpoint_sha256":chexzero.checkpoint_hash,"coverage":chexzero.coverage,
                  "frozen":True,"whole_image_only":True,"receives_dev_final_labels":False,
                  "role":["whole-image global evidence","text margins","image-text similarity","recoverable global reference"]}
    atomic_json(OUT/"frozen_chexzero_reference.json",chexzero_ref)
log_event("CheXzero immutable reference persisted",level="OK",step="8.3",checkpoint_hash=chexzero.checkpoint_hash[:16],coverage=f"{chexzero.coverage:.2%}")
cell_end(_CELL_T0,"8.3","Instantiate and persist the immutable CheXzero reference")


[+  1171.9s] [CELL   ] [8.3] BEGIN CELL — Instantiate and persist the immutable CheXzero reference
[+  1171.9s] [STEP   ] [8.3] START — Instantiate and persist immutable CheXzero reference
[+  1171.9s] [STEP   ] [8.2] Initializing frozen CheXzero ViT-B/32 | checkpoint=/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/best_64_5e-05_original_22000_0.864.pt
[+  1171.9s] [STEP   ] [2.2] Hashing file | file=/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/best_64_5e-05_original_22000_0.864.pt | size_mb=337.2
[+  1176.1s] [OK     ] [2.2] File hash complete | file=best_64_5e-05_original_22000_0.864.pt | sha256=567a2efbabc6a023 | elapsed=4.23s
[+  1176.1s] [STEP   ] [8.2] Creating CheXzero ViT-B/32 architecture


[+  1177.3s] [STEP   ] [8.2] Loading CheXzero checkpoint tensors
[+  1177.5s] [INFO   ] [8.2] CheXzero checkpoint coverage calculated | coverage=100.00% | matched_tensors=302 | model_tensors=302
[+  1177.9s] [STEP   ] [8.2] Building frozen CheXzero pathology text anchors | classes=5
[+  1178.2s] [PROGRESS] [8.2] CheXzero text-anchor encoding 1/5 (20.0%) | pathology=Atelectasis
[+  1178.4s] [PROGRESS] [8.2] CheXzero text-anchor encoding 2/5 (40.0%) | pathology=Cardiomegaly
[+  1178.5s] [PROGRESS] [8.2] CheXzero text-anchor encoding 3/5 (60.0%) | pathology=Consolidation
[+  1178.5s] [PROGRESS] [8.2] CheXzero text-anchor encoding 4/5 (80.0%) | pathology=Edema
[+  1178.6s] [PROGRESS] [8.2] CheXzero text-anchor encoding 5/5 (100.0%) | pathology=Pleural Effusion
[+  1178.6s] [OK     ] [8.2] Frozen CheXzero initialized | coverage=100.00% | elapsed=6.78s | gpu0_alloc_mb=1600.6 | gpu0_reserved_mb=1664.0 | gpu1_alloc_mb=0.0 | gpu1_reserved_mb=0.0
[+  1178.6s] [SAVE   ] [2.2] Saving JSON | file=/

In [25]:
_CELL_T0=cell_begin("8.3","Instantiate and persist the immutable CheXzero reference")
assert all(not p.requires_grad for p in chexzero.model.parameters()), "CheXzero must remain frozen."
assert chexzero_ref["whole_image_only"] is True
assert chexzero_ref["receives_dev_final_labels"] is False
log_event("CheXzero freeze/role audit passed",level="OK",step="8.3",frozen_parameters=sum(p.numel() for p in chexzero.model.parameters()))
cell_end(_CELL_T0,"8.3","Instantiate and persist the immutable CheXzero reference")


[+  1178.8s] [CELL   ] [8.3] BEGIN CELL — Instantiate and persist the immutable CheXzero reference
[+  1178.8s] [OK     ] [8.3] CheXzero freeze/role audit passed | frozen_parameters=151277313
[+  1178.8s] [CELL   ] [8.3] END CELL — Instantiate and persist the immutable CheXzero reference | elapsed=0.00s


## 9. Losses, image-level patch pooling, and metrics
### 9.1 Masked BCE and one-vs-rest ranking losses


In [26]:
_CELL_T0=cell_begin("9.1","Masked BCE and one-vs-rest ranking losses")
def masked_bce(logits,y):
    m=torch.isfinite(y); t=torch.nan_to_num(y,nan=0.0)
    raw=F.binary_cross_entropy_with_logits(logits,t,reduction="none")
    return (raw*m.float()).sum()/m.float().sum().clamp_min(1)

def pairwise_ovr_rank(logits,y,margin=1.0):
    losses=[]
    for c in range(logits.shape[1]):
        known=torch.isfinite(y[:,c]); pos=known&(y[:,c]>=.5); neg=known&(y[:,c]<.5)
        if pos.any() and neg.any():
            d=logits[pos,c][:,None]-logits[neg,c][None,:]
            losses.append(F.softplus(margin-d).mean())
    return torch.stack(losses).mean() if losses else logits.sum()*0
cell_end(_CELL_T0,"9.1","Masked BCE and one-vs-rest ranking losses")


[+  1179.0s] [CELL   ] [9.1] BEGIN CELL — Masked BCE and one-vs-rest ranking losses
[+  1179.0s] [CELL   ] [9.1] END CELL — Masked BCE and one-vs-rest ranking losses | elapsed=0.00s


### 9.2 Equal-image-weight MIL pooling and macro metrics


In [27]:
_CELL_T0=cell_begin("9.2","Equal-image-weight MIL pooling and macro metrics")
def pool_by_owner(v,owner,n_images):
    # log-mean-exp MIL pooling; each IMAGE gets exactly one output regardless of patch count
    out=[]
    for i in range(n_images):
        q=v[owner.eq(i)]
        if len(q)==0: raise RuntimeError(f"Patch pooling received no patches for image index {i}.")
        out.append(torch.logsumexp(q,dim=0)-math.log(len(q)))
    return torch.stack(out)

def macro_metrics(y,s):
    y=np.asarray(y,float); s=np.asarray(s,float); auc=[]; ap=[]; per={}
    if y.ndim!=2 or s.ndim!=2 or y.shape!=s.shape or y.shape[1]!=N_CLASSES:
        raise RuntimeError(f"Metric shape mismatch: labels={y.shape}, scores={s.shape}, expected N_CLASSES={N_CLASSES}")
    for c,name in enumerate(CLASSES):
        m=np.isfinite(y[:,c])
        if m.sum() and len(np.unique(y[m,c]))==2:
            a=roc_auc_score(y[m,c],s[m,c]); p=average_precision_score(y[m,c],s[m,c])
        else: a=p=np.nan
        auc.append(a); ap.append(p); per[name]={"auroc":a,"auprc":p,"known":int(m.sum()),"positives":int(np.nansum(y[m,c])) if m.any() else 0}
    finite_auc=[x for x in auc if np.isfinite(x)]; finite_ap=[x for x in ap if np.isfinite(x)]
    return {"macro_auroc":float(np.mean(finite_auc)) if finite_auc else float("nan"),
            "macro_auprc":float(np.mean(finite_ap)) if finite_ap else float("nan"),"per_class":per}

def selection_score(metrics):
    """Full protocol selects by macro AUROC. Smoke falls back to AUPRC only when AUROC is undefined."""
    a=float(metrics.get("macro_auroc",np.nan))
    if np.isfinite(a): return a,"macro_auroc"
    p=float(metrics.get("macro_auprc",np.nan))
    if smoke_enabled() and np.isfinite(p):
        log_event("Smoke selection fallback: macro AUROC undefined; using macro AUPRC only to keep pipeline executable",level="WARN",step="9.2",macro_auprc=round(p,6))
        return p,"macro_auprc_smoke_fallback"
    return -np.inf,"undefined"
cell_end(_CELL_T0,"9.2","Equal-image-weight MIL pooling and macro metrics")


[+  1179.1s] [CELL   ] [9.2] BEGIN CELL — Equal-image-weight MIL pooling and macro metrics
[+  1179.1s] [CELL   ] [9.2] END CELL — Equal-image-weight MIL pooling and macro metrics | elapsed=0.00s


### 9.3 Optimizer and scheduler factory


In [28]:
_CELL_T0=cell_begin("9.3","Optimizer and scheduler factory")
_DP_ANNOUNCED=False

def forward_core(model,x):
    global _DP_ANNOUNCED
    if data_parallel_enabled() and x.shape[0]>=torch.cuda.device_count():
        if not _DP_ANNOUNCED:
            log_event("Full-run multi-GPU forward enabled",level="OK",step="9.3",device_ids=list(range(torch.cuda.device_count())))
            _DP_ANNOUNCED=True
        return nn.parallel.data_parallel(model,x,device_ids=list(range(torch.cuda.device_count())),output_device=0)
    return model(x)

def optimizer_for(model):
    enc=[p for p in model.encoder.parameters() if p.requires_grad]
    head=[p for p in model.head.parameters() if p.requires_grad]
    opt=torch.optim.AdamW([{"params":enc,"lr":cfg.backbone_lr},{"params":head,"lr":cfg.head_lr}],weight_decay=cfg.weight_decay)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,effective_nih_epochs()))
    log_event("Optimizer/scheduler created",level="INFO",step="9.3",encoder_params=sum(p.numel() for p in enc),head_params=sum(p.numel() for p in head),
              backbone_lr=cfg.backbone_lr,head_lr=cfg.head_lr,weight_decay=cfg.weight_decay,scheduler_Tmax=sched.T_max)
    return opt,sched
cell_end(_CELL_T0,"9.3","Optimizer and scheduler factory")


[+  1179.2s] [CELL   ] [9.3] BEGIN CELL — Optimizer and scheduler factory
[+  1179.2s] [CELL   ] [9.3] END CELL — Optimizer and scheduler factory | elapsed=0.00s


## 10. Evaluation utilities
### 10.1 NIH held-out patient evaluation


In [29]:
_CELL_T0=cell_begin("10.1","NIH held-out patient evaluation")
@torch.inference_mode()
def evaluate_nih(model,df,kind):
    t0=time.time(); model.eval(); ys=[]; ss=[]; workers=effective_num_workers()
    if len(df)==0: raise RuntimeError("NIH evaluation split is empty.")
    if kind=="whole":
        dl=DataLoader(NIHWholeDS(df,False),batch_size=cfg.batch_images,shuffle=False,num_workers=workers,pin_memory=True)
    else:
        dl=DataLoader(NIHPatchBagDS(df,False),batch_size=cfg.batch_images,shuffle=False,num_workers=workers,pin_memory=True,collate_fn=collate_patch_bags)
    log_event("NIH evaluation started",level="EVAL",step="10.1",branch=kind,images=len(df),batches=len(dl),workers=workers)
    for bi,batch in enumerate(dl,1):
        if kind=="whole":
            x,y,*_=batch; o=forward_core(model,x.to(DEVICE,non_blocking=True)); ss.append(o["logits"].float().cpu()); ys.append(y)
        else:
            x,owner,y,*_=batch; o=forward_core(model,x.to(DEVICE,non_blocking=True)); pooled=pool_by_owner(o["logits"],owner.to(DEVICE),len(y))
            ss.append(pooled.float().cpu()); ys.append(y)
        if bi%max(1,batch_log_every())==0 or bi==len(dl):
            progress_log("10.1",f"NIH {kind} evaluation",bi,len(dl),**gpu_snapshot())
    Y=torch.cat(ys).numpy(); S=torch.cat(ss).numpy(); met=macro_metrics(Y,S)
    log_event("NIH evaluation complete",level="OK",step="10.1",branch=kind,macro_auroc=met["macro_auroc"],macro_auprc=met["macro_auprc"],elapsed=f"{time.time()-t0:.2f}s")
    return met
cell_end(_CELL_T0,"10.1","NIH held-out patient evaluation")


[+  1179.3s] [CELL   ] [10.1] BEGIN CELL — NIH held-out patient evaluation
[+  1179.3s] [CELL   ] [10.1] END CELL — NIH held-out patient evaluation | elapsed=0.00s


## 11. T0 — NIH source training
### 11.1 Exact epoch checkpoint state and deterministic loaders


In [30]:
_CELL_T0=cell_begin("11.1","Exact epoch checkpoint state and deterministic loaders")
def ckpt_path(branch,phase): return OUT/f"resume_{branch}_{phase}.pt"

def save_resume(branch,phase,model,opt,sched,scaler,next_epoch,best,best_state,history):
    payload={"pipeline_version":PIPELINE_VERSION,"branch":branch,"phase":phase,"model":model.state_dict(),"optimizer":opt.state_dict(),
        "scheduler":sched.state_dict(),"scaler":scaler.state_dict(),"next_epoch":next_epoch,"best":best,"best_state":best_state,
        "history":history,"rng":rng_state(),"nih_manifest_hash":canonical_hash(nih_manifest.to_dict("records"))}
    atomic_torch(ckpt_path(branch,phase),payload)
    log_event("Exact resume checkpoint committed",level="OK",step="11.1",branch=branch,phase=phase,next_epoch=next_epoch,best=best)

def maybe_resume(branch,phase,model,opt,sched,scaler):
    p=ckpt_path(branch,phase)
    if cfg.resume and p.exists():
        log_event("Resume checkpoint found",level="RESUME",step="11.1",branch=branch,phase=phase,file=str(p))
        q=torch.load(p,map_location=DEVICE,weights_only=False)
        if q.get("pipeline_version")!=PIPELINE_VERSION:
            raise RuntimeError(f"Refusing incompatible resume checkpoint {p.name}: {q.get('pipeline_version')} != {PIPELINE_VERSION}")
        expected=canonical_hash(nih_manifest.to_dict("records"))
        if q.get("nih_manifest_hash")!=expected:
            raise RuntimeError(f"Refusing resume checkpoint {p.name}: NIH manifest hash changed.")
        model.load_state_dict(q["model"]); opt.load_state_dict(q["optimizer"])
        sched.load_state_dict(q["scheduler"]); scaler.load_state_dict(q["scaler"]); restore_rng(q["rng"])
        log_event("Resume checkpoint restored exactly",level="OK",step="11.1",branch=branch,phase=phase,next_epoch=q["next_epoch"],history_rows=len(q["history"]))
        return q["next_epoch"],q["best"],q["best_state"],q["history"]
    log_event("No compatible resume checkpoint; starting phase from epoch 0",level="INFO",step="11.1",branch=branch,phase=phase)
    return 0,-np.inf,None,[]

def epoch_order(n,seed): return np.random.default_rng(seed).permutation(n).tolist()

def ordered_loader(ds,order,kind):
    sub=torch.utils.data.Subset(ds,order)
    kwargs=dict(batch_size=cfg.batch_images,shuffle=False,num_workers=effective_num_workers(),pin_memory=True)
    if kind=="patch": kwargs["collate_fn"]=collate_patch_bags
    return DataLoader(sub,**kwargs)
cell_end(_CELL_T0,"11.1","Exact epoch checkpoint state and deterministic loaders")


[+  1179.4s] [CELL   ] [11.1] BEGIN CELL — Exact epoch checkpoint state and deterministic loaders
[+  1179.5s] [CELL   ] [11.1] END CELL — Exact epoch checkpoint state and deterministic loaders | elapsed=0.00s


### 11.2 Independent whole/patch source-training routine


In [31]:
_CELL_T0=cell_begin("11.2","Independent whole/patch source-training routine")
def train_nih_branch(model,kind):
    t_phase=time.time(); branch=kind
    tr=nih_manifest[nih_manifest.source_split.eq("train")].reset_index(drop=True)
    va=nih_manifest[nih_manifest.source_split.eq("validation")].reset_index(drop=True)
    if len(tr)==0 or len(va)==0: raise RuntimeError(f"NIH {kind}: train/validation split cannot be empty.")
    ds=NIHWholeDS(tr,True) if kind=="whole" else NIHPatchBagDS(tr,True)
    opt,sched=optimizer_for(model); scaler=torch.amp.GradScaler("cuda",enabled=cfg.amp)
    start,best,best_state,hist=maybe_resume(branch,"nih",model,opt,sched,scaler)
    log_event("NIH source training phase started",level="TRAIN",step="11.2",branch=kind,train_images=len(tr),validation_images=len(va),
              start_epoch=start+1,total_epochs=effective_nih_epochs(),amp=cfg.amp,workers=effective_num_workers(),data_parallel=data_parallel_enabled())
    last=None
    for ep in range(start,effective_nih_epochs()):
        t0=time.time(); model.train(); run=0.; nb=0
        dl=ordered_loader(ds,epoch_order(len(ds),cfg.seed+1000*ep+(0 if kind=="whole" else 50)),kind)
        log_event("NIH epoch started",level="TRAIN",step="11.2",branch=kind,epoch=f"{ep+1}/{effective_nih_epochs()}",batches=len(dl))
        for bi,batch in enumerate(dl,1):
            bt=time.time(); opt.zero_grad(set_to_none=True)
            if kind=="whole":
                x,y,*_=batch; x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE)
                with torch.autocast("cuda",enabled=cfg.amp):
                    o=forward_core(model,x); logits=o["logits"]
                    loss=masked_bce(logits,y)+cfg.source_auc_rank_weight*pairwise_ovr_rank(logits,y)+cfg.orth_weight*orthogonality_loss(o["zc"],o["zs"])
            else:
                x,owner,y,*_=batch; x=x.to(DEVICE,non_blocking=True); owner=owner.to(DEVICE); y=y.to(DEVICE)
                with torch.autocast("cuda",enabled=cfg.amp):
                    o=forward_core(model,x); logits=pool_by_owner(o["logits"],owner,len(y)); zc=pool_by_owner(o["zc"],owner,len(y)); zs=pool_by_owner(o["zs"],owner,len(y))
                    loss=masked_bce(logits,y)+cfg.source_auc_rank_weight*pairwise_ovr_rank(logits,y)+cfg.orth_weight*orthogonality_loss(zc,zs)
            if not torch.isfinite(loss): raise FloatingPointError(f"NIH {kind} produced non-finite loss at epoch {ep+1}, batch {bi}: {loss.item()}")
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            grad_norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],5.0)
            scaler.step(opt); scaler.update(); run+=float(loss.detach()); nb+=1
            if bi%max(1,batch_log_every())==0 or bi==len(dl):
                progress_log("11.2",f"NIH {kind} epoch {ep+1}",bi,len(dl),loss=round(float(loss.detach()),6),
                             avg_loss=round(run/nb,6),grad_norm=round(float(grad_norm),4),batch_s=round(time.time()-bt,3),**gpu_snapshot())
        sched.step(); met=evaluate_nih(model,va,kind); score,score_name=selection_score(met)
        row={"branch":kind,"phase":"nih","epoch":ep+1,"loss":run/max(1,nb),"selection_score":score,"selection_metric":score_name,
             **{k:v for k,v in met.items() if k!="per_class"}}
        hist.append(row)
        is_best=(best_state is None) or (score>best)
        if is_best:
            best=score; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            log_event("New best NIH checkpoint selected",level="BEST",step="11.2",branch=kind,epoch=ep+1,metric=score_name,score=score)
        last=time.time()-t0
        log_event("NIH epoch complete",level="OK",step="11.2",branch=kind,epoch=ep+1,avg_loss=round(row["loss"],6),
                  macro_auroc=met["macro_auroc"],macro_auprc=met["macro_auprc"],elapsed=f"{last:.2f}s")
        save_resume(branch,"nih",model,opt,sched,scaler,ep+1,best,best_state,hist)
        if ep+1<effective_nih_epochs() and should_stop_before_next_unit(last):
            save_status("partial",next=f"{kind}:NIH epoch {ep+2}"); return False,hist
    if best_state is None:
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        log_event("No finite selection metric; retaining final NIH state for smoke executability",level="WARN",step="11.2",branch=kind)
    model.load_state_dict(best_state)
    finite_auc=[r["macro_auroc"] for r in hist if np.isfinite(r.get("macro_auroc",np.nan))]
    best_auc=max(finite_auc) if finite_auc else float("nan")
    atomic_torch(OUT/f"T0_NIH_{kind}_vit.pt",{"pipeline_version":PIPELINE_VERSION,"model":model.state_dict(),"best_validation_auroc":best_auc,
        "best_selection_score":best,"history":hist,"imagenet_vit_sha256":IMAGENET_VIT_HASH,
        "trainable_parameters":sum(p.numel() for p in model.parameters() if p.requires_grad),"run_mode":cfg.run_mode,"smoke_test":smoke_enabled()})
    log_event("T0 NIH branch complete",level="OK",step="11.2",branch=kind,checkpoint=f"T0_NIH_{kind}_vit.pt",elapsed=f"{time.time()-t_phase:.2f}s")
    return True,hist
cell_end(_CELL_T0,"11.2","Independent whole/patch source-training routine")


[+  1179.6s] [CELL   ] [11.2] BEGIN CELL — Independent whole/patch source-training routine
[+  1179.6s] [CELL   ] [11.2] END CELL — Independent whole/patch source-training routine | elapsed=0.00s


## 13. Frozen CheXzero feature cache
### 13.1 Cache whole-image embeddings, text margins, and positive similarity


In [32]:
_CELL_T0=cell_begin("13.1","Cache whole-image embeddings, text margins, and positive similarity")
def cache_chexzero_reference():
    if not cfg.cache_chexzero:
        log_event("Frozen CheXzero feature cache disabled by configuration",level="INFO",step="13.1")
        return None
    p=OUT/"chexzero_matched_nih_cache.pt"
    if p.exists():
        log_event("Existing CheXzero feature cache found; validating",level="LOAD",step="13.1",file=str(p))
        q=torch.load(p,map_location="cpu",weights_only=False)
        valid=(q.get("checkpoint_sha256")==chexzero.checkpoint_hash and q.get("image_id")==nih_manifest.image_id.tolist())
        if valid:
            log_event("CheXzero feature cache hit",level="OK",step="13.1",images=len(q["image_id"]),shape=tuple(q["features"].shape))
            return q
        log_event("Existing CheXzero cache is stale; rebuilding",level="WARN",step="13.1")
    paths=nih_manifest.whole_image_path.tolist(); z=chexzero.encode_paths(paths,batch=64)
    log_event("Computing CheXzero text margins",level="STEP",step="13.1",images=len(paths))
    margins=chexzero.margins(z)
    log_event("Computing CheXzero positive similarities",level="STEP",step="13.1",images=len(paths))
    pos=chexzero.positive_similarity(z)
    payload={"pipeline_version":PIPELINE_VERSION,"image_id":nih_manifest.image_id.tolist(),"features":z,"text_margin":margins,
             "positive_similarity":pos,"checkpoint_sha256":chexzero.checkpoint_hash}
    atomic_torch(p,payload)
    log_event("Frozen CheXzero feature cache complete",level="OK",step="13.1",features=tuple(z.shape),margins=tuple(margins.shape),similarity=tuple(pos.shape))
    return payload

with timed_step("13.1","Frozen CheXzero feature-cache stage"):
    _ = cache_chexzero_reference()
cell_end(_CELL_T0,"13.1","Cache whole-image embeddings, text margins, and positive similarity")


[+  1179.7s] [CELL   ] [13.1] BEGIN CELL — Cache whole-image embeddings, text margins, and positive similarity
[+  1179.7s] [STEP   ] [13.1] START — Frozen CheXzero feature-cache stage
[+  1179.7s] [STEP   ] [13.1] Encoding frozen CheXzero whole-image features | images=20746 | batches=325 | batch_size=64 | workers=2
[+  1202.0s] [PROGRESS] [13.1] CheXzero image encoding 25/325 (7.7%) | encoded=1600 | gpu0_alloc_mb=1245.6 | gpu0_reserved_mb=1702.0 | gpu1_alloc_mb=0.0 | gpu1_reserved_mb=0.0
[+  1223.0s] [PROGRESS] [13.1] CheXzero image encoding 50/325 (15.4%) | encoded=3200 | gpu0_alloc_mb=1245.6 | gpu0_reserved_mb=1702.0 | gpu1_alloc_mb=0.0 | gpu1_reserved_mb=0.0
[+  1245.3s] [PROGRESS] [13.1] CheXzero image encoding 75/325 (23.1%) | encoded=4800 | gpu0_alloc_mb=1245.6 | gpu0_reserved_mb=1702.0 | gpu1_alloc_mb=0.0 | gpu1_reserved_mb=0.0
[+  1266.4s] [PROGRESS] [13.1] CheXzero image encoding 100/325 (30.8%) | encoded=6400 | gpu0_alloc_mb=1245.6 | gpu0_reserved_mb=1702.0 | gpu1_alloc_mb=0

### 13.2 Smoke/full preflight gate

In [33]:
_CELL_T0=cell_begin("13.2","NIH-only preflight gate")
n_nih=len(nih_manifest)
if smoke_enabled(): assert 1<=n_nih<=cfg.smoke_nih_images
else: assert n_nih==cfg.expected_cohort_size
assert len(nih_manifest[nih_manifest.source_split.eq("train")])>0
assert len(nih_manifest[nih_manifest.source_split.eq("validation")])>0
assert nih_manifest[list(LABEL_COLS)].sum(axis=1).eq(1).all()
assert all(not p.requires_grad for p in chexzero.model.parameters())
log_event("NIH-only preflight passed",level="OK",step="13.2",nih_images=n_nih,nih_epochs=effective_nih_epochs(),
          frozen_chexzero=True,chexpert_preadaptation=False,output=str(OUT))
cell_end(_CELL_T0,"13.2","NIH-only preflight gate")

[+  1451.7s] [CELL   ] [13.2] BEGIN CELL — NIH-only preflight gate
[+  1451.7s] [OK     ] [13.2] NIH-only preflight passed | nih_images=20746 | nih_epochs=10 | frozen_chexzero=True | chexpert_preadaptation=False | output=/kaggle/working/qfid_matched_stage1_nih_only
[+  1451.7s] [CELL   ] [13.2] END CELL — NIH-only preflight gate | elapsed=0.02s


## 14. Resumable Stage-1 execution

In [34]:
_CELL_T0=cell_begin("14","Resumable Stage-1 execution")
log_event("Stage-1 NIH-only execution begins",level="RUN",step="14",next_unit="whole NIH")
with timed_step("14","UNIT 1/2 T0 whole-image NIH training"):
    ok_whole_nih,h_whole=train_nih_branch(whole,"whole")
if not ok_whole_nih: raise SystemExit("Graceful runtime stop after completed whole-NIH epoch. Rerun to continue.")
if should_stop_before_next_unit(): save_status("partial",next="patch NIH"); raise SystemExit("Graceful runtime stop. Rerun to continue at patch NIH.")
with timed_step("14","UNIT 2/2 T0 patch NIH training"):
    ok_patch_nih,h_patch=train_nih_branch(patch,"patch")
if not ok_patch_nih: raise SystemExit("Graceful runtime stop after completed patch-NIH epoch. Rerun to continue.")
log_event("Both NIH training units completed",level="OK",step="14",**gpu_snapshot())
cell_end(_CELL_T0,"14","Resumable Stage-1 execution")

[+  1451.8s] [CELL   ] [14] BEGIN CELL — Resumable Stage-1 execution
[+  1451.8s] [RUN    ] [14] Stage-1 NIH-only execution begins | next_unit=whole NIH
[+  1451.8s] [STEP   ] [14] START — UNIT 1/2 T0 whole-image NIH training
[+  1451.9s] [INFO   ] [9.3] Optimizer/scheduler created | encoder_params=14177280 | head_params=265861 | backbone_lr=1e-05 | head_lr=0.0003 | weight_decay=0.01 | scheduler_Tmax=10
[+  1451.9s] [INFO   ] [11.1] No compatible resume checkpoint; starting phase from epoch 0 | branch=whole | phase=nih
[+  1451.9s] [TRAIN  ] [11.2] NIH source training phase started | branch=whole | train_images=16472 | validation_images=4274 | start_epoch=1 | total_epochs=10 | amp=True | workers=2 | data_parallel=True
[+  1451.9s] [TRAIN  ] [11.2] NIH epoch started | branch=whole | epoch=1/10 | batches=1030
[+  1452.3s] [OK     ] [9.3] Full-run multi-GPU forward enabled | device_ids=[0, 1]
[+  1458.8s] [PROGRESS] [11.2] NIH whole epoch 1 25/1030 (2.4%) | loss=0.687865 | avg_loss=0.7074

## 15. Validation, histories, and Stage-1 bundle

In [35]:
_CELL_T0=cell_begin("15.1","Evaluate selected NIH checkpoints")
def metrics_from_checkpoint(model,kind,path):
    path=Path(path); payload=torch.load(path,map_location=DEVICE,weights_only=False); model.load_state_dict(payload["model"])
    result={"NIH_validation":evaluate_nih(model,nih_manifest[nih_manifest.source_split.eq("validation")],kind)}
    log_event("Checkpoint validation complete",level="OK",step="15.1",checkpoint=path.name,
              summary={k:{"macro_auroc":v["macro_auroc"],"macro_auprc":v["macro_auprc"]} for k,v in result.items()})
    return result
validation={"T0_whole":metrics_from_checkpoint(whole,"whole",OUT/"T0_NIH_whole_vit.pt"),
            "T0_patch":metrics_from_checkpoint(patch,"patch",OUT/"T0_NIH_patch_vit.pt")}
cell_end(_CELL_T0,"15.1","Evaluate selected NIH checkpoints")

[+ 16306.3s] [CELL   ] [15.1] BEGIN CELL — Evaluate selected NIH checkpoints
[+ 16306.6s] [EVAL   ] [10.1] NIH evaluation started | branch=whole | images=4274 | batches=268 | workers=2
[+ 16312.3s] [PROGRESS] [10.1] NIH whole evaluation 25/268 (9.3%) | gpu0_alloc_mb=1723.6 | gpu0_reserved_mb=5474.0 | gpu1_alloc_mb=18.2 | gpu1_reserved_mb=3530.0
[+ 16317.2s] [PROGRESS] [10.1] NIH whole evaluation 50/268 (18.7%) | gpu0_alloc_mb=1723.6 | gpu0_reserved_mb=5474.0 | gpu1_alloc_mb=18.2 | gpu1_reserved_mb=3530.0
[+ 16322.7s] [PROGRESS] [10.1] NIH whole evaluation 75/268 (28.0%) | gpu0_alloc_mb=1723.6 | gpu0_reserved_mb=5474.0 | gpu1_alloc_mb=18.2 | gpu1_reserved_mb=3530.0
[+ 16327.7s] [PROGRESS] [10.1] NIH whole evaluation 100/268 (37.3%) | gpu0_alloc_mb=1723.6 | gpu0_reserved_mb=5474.0 | gpu1_alloc_mb=18.2 | gpu1_reserved_mb=3530.0
[+ 16332.7s] [PROGRESS] [10.1] NIH whole evaluation 125/268 (46.6%) | gpu0_alloc_mb=1723.6 | gpu0_reserved_mb=5474.0 | gpu1_alloc_mb=18.2 | gpu1_reserved_mb=3530.0

### 15.2 Save NIH training histories

In [36]:
_CELL_T0=cell_begin("15.2","Save NIH training histories")
for name,frame in {"training_history_whole.csv":pd.DataFrame(h_whole),"training_history_patch.csv":pd.DataFrame(h_patch)}.items():
    frame.to_csv(OUT/name,index=False); log_event("Saved training history",level="OK",step="15.2",file=name,rows=len(frame))
cell_end(_CELL_T0,"15.2","Save NIH training histories")

[+ 16815.4s] [CELL   ] [15.2] BEGIN CELL — Save NIH training histories
[+ 16815.4s] [OK     ] [15.2] Saved training history | file=training_history_whole.csv | rows=10
[+ 16815.4s] [OK     ] [15.2] Saved training history | file=training_history_patch.csv | rows=10
[+ 16815.4s] [CELL   ] [15.2] END CELL — Save NIH training histories | elapsed=0.01s


### 15.3 Save complete per-pathology validation metrics for T0 and T1


In [37]:
_CELL_T0=cell_begin("15.3","Save complete per-pathology validation metrics for T0 and T1")
rows=[]
for checkpoint,stages in validation.items():
    branch="whole" if checkpoint.endswith("whole") else "patch"
    tier="T0"
    for stage,met in stages.items():
        for cls,v in met["per_class"].items():
            rows.append({"tier":tier,"branch":branch,"stage":stage,"pathology":cls,**v})
metrics_df=pd.DataFrame(rows)
metrics_df.to_csv(OUT/"per_class_validation_metrics.csv",index=False)
log_event("Saved per-pathology validation metrics",level="OK",step="15.3",rows=len(metrics_df),file="per_class_validation_metrics.csv")
cell_end(_CELL_T0,"15.3","Save complete per-pathology validation metrics for T0 and T1")


[+ 16815.7s] [CELL   ] [15.3] BEGIN CELL — Save complete per-pathology validation metrics for T0 and T1
[+ 16815.7s] [OK     ] [15.3] Saved per-pathology validation metrics | rows=10 | file=per_class_validation_metrics.csv
[+ 16815.7s] [CELL   ] [15.3] END CELL — Save complete per-pathology validation metrics for T0 and T1 | elapsed=0.00s


### 15.4 NIH dataset/split hashes and patient manifests

In [38]:
_CELL_T0=cell_begin("15.4","NIH dataset/split hashes and patient manifests")
split_hashes={"nih_manifest":canonical_hash(nih_manifest.to_dict("records")),
 "nih_train":canonical_hash(nih_manifest[nih_manifest.source_split.eq("train")].to_dict("records")),
 "nih_validation":canonical_hash(nih_manifest[nih_manifest.source_split.eq("validation")].to_dict("records"))}
patient_manifests={"nih_train":sorted(nih_manifest.loc[nih_manifest.source_split.eq("train"),"patient_id"].unique().tolist()),
 "nih_validation":sorted(nih_manifest.loc[nih_manifest.source_split.eq("validation"),"patient_id"].unique().tolist())}
log_event("NIH hashes and patient manifests ready",level="OK",step="15.4",hashes={k:v[:12] for k,v in split_hashes.items()},patient_counts={k:len(v) for k,v in patient_manifests.items()})
cell_end(_CELL_T0,"15.4","NIH dataset/split hashes and patient manifests")

[+ 16816.0s] [CELL   ] [15.4] BEGIN CELL — NIH dataset/split hashes and patient manifests
[+ 16816.6s] [OK     ] [15.4] NIH hashes and patient manifests ready | hashes={'nih_manifest': '1faa73c4839a', 'nih_train': 'bdd77bbfc7c3', 'nih_validation': '6b367a7e8f07'} | patient_counts={'nih_train': 5802, 'nih_validation': 1474}
[+ 16816.6s] [CELL   ] [15.4] END CELL — NIH dataset/split hashes and patient manifests | elapsed=0.60s


### 15.5 Assemble the NIH Stage-1 bundle

In [39]:
_CELL_T0=cell_begin("15.5","Assemble NIH Stage-1 bundle")
model_files={"T0_whole":OUT/"T0_NIH_whole_vit.pt","T0_patch":OUT/"T0_NIH_patch_vit.pt"}
loaded_models={key:torch.load(path,map_location="cpu",weights_only=False) for key,path in model_files.items()}
bundle={"pipeline_version":PIPELINE_VERSION,"protocol":"QFID-CXR matched NIH exactly-one-of-five; no CheXpert pre-adaptation",
 "execution":{"run_mode":cfg.run_mode,"smoke_test":smoke_enabled(),"nih_images":len(nih_manifest),"nih_epochs":effective_nih_epochs(),
              "chexpert_preadaptation":False},"classes":CLASSES,
 "label_policy":{"NIH":"official exactly-one-positive-of-five ChestX-ray14 image labels"},"models":loaded_models,
 "imagenet_vit_checkpoint_hash":IMAGENET_VIT_HASH,"chexzero_checkpoint_hash":chexzero.checkpoint_hash,
 "chexzero_reference":chexzero_ref,"chexzero_cache":"chexzero_matched_nih_cache.pt",
 "split_hashes":split_hashes,"patient_manifests":patient_manifests,
 "trainable_parameter_counts":{"whole":sum(p.numel() for p in whole.parameters() if p.requires_grad),"patch":sum(p.numel() for p in patch.parameters() if p.requires_grad)},
 "random_seeds":{"global":cfg.seed},"resume_state":{"unit":"epoch","checkpoint_files":[str(x.name) for x in OUT.glob("resume_*.pt")]},
 "validation":validation,"config":asdict(cfg)}
atomic_torch(OUT/"QFID_MATCHED_STAGE1_BUNDLE.pt",bundle)
manifest={"pipeline_version":PIPELINE_VERSION,"bundle":"QFID_MATCHED_STAGE1_BUNDLE.pt","models":list(model_files),
          "nih_images":len(nih_manifest),"official_exact_one_size":cfg.expected_cohort_size,"chexpert_preadaptation":False,
          "frozen_chexzero_retained":True,"complete_pack_files":[p.name for p in stage_paths().values()]}
atomic_json(OUT/"QFID_MATCHED_STAGE1_MANIFEST.json",manifest)
cell_end(_CELL_T0,"15.5","Assemble NIH Stage-1 bundle")

[+ 16816.9s] [CELL   ] [15.5] BEGIN CELL — Assemble NIH Stage-1 bundle
[+ 16817.4s] [SAVE   ] [2.2] Saving torch checkpoint | file=/kaggle/working/qfid_matched_stage1_nih_only/QFID_MATCHED_STAGE1_BUNDLE.pt
[+ 16818.0s] [OK     ] [2.2] Saved torch checkpoint | file=QFID_MATCHED_STAGE1_BUNDLE.pt | size_mb=656.9 | elapsed=0.62s
[+ 16818.0s] [OK     ] [3.3] Resolved exactly three complete Stage-3 pack paths | causal=/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/stage3_causal_complete.pt | spur_in=/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/stage3_spur_in_complete.pt | spur_out=/kaggle/input/datasets/anikataf/recovery-patches-14thsept/output/nih_remainder/stage3_spur_out_complete.pt
[+ 16818.0s] [SAVE   ] [2.2] Saving JSON | file=/kaggle/working/qfid_matched_stage1_nih_only/QFID_MATCHED_STAGE1_MANIFEST.json
[+ 16818.0s] [OK     ] [2.2] Saved JSON | file=QFID_MATCHED_STAGE1_MANIFEST.json | size_kb=0.4 | elapsed=0.00s
[+ 1681

### 15.6 Final artifact-completeness gate

In [40]:
_CELL_T0=cell_begin("15.6","Final artifact-completeness gate")
REQUIRED_OUTPUTS=["matched_nih_manifest.csv","matched_nih_train.csv","matched_nih_validation.csv","matched_nih_audit.json",
 "T0_NIH_whole_vit.pt","T0_NIH_patch_vit.pt","frozen_chexzero_reference.json","chexzero_matched_nih_cache.pt",
 "training_history_whole.csv","training_history_patch.csv","per_class_validation_metrics.csv",
 "QFID_MATCHED_STAGE1_BUNDLE.pt","QFID_MATCHED_STAGE1_MANIFEST.json"]
missing=[name for name in REQUIRED_OUTPUTS if not (OUT/name).exists()]
if missing:
    save_status("partial",next="artifact completeness",missing_outputs=missing)
    raise RuntimeError(f"Stage-1 incomplete; missing required outputs: {missing}")
atomic_json(OUT/"execution_events.json",RUN_EVENTS); save_status("complete",next=None)
log_event("STAGE 1 NIH-ONLY COMPLETE",level="SUCCESS",step="15.6",output=str(OUT),required_artifacts=f"{len(REQUIRED_OUTPUTS)}/{len(REQUIRED_OUTPUTS)}",total_elapsed=f"{time.time()-START_TIME:.1f}s",**gpu_snapshot())
cell_end(_CELL_T0,"15.6","Final artifact-completeness gate")

[+ 16818.3s] [CELL   ] [15.6] BEGIN CELL — Final artifact-completeness gate
[+ 16818.3s] [SAVE   ] [2.2] Saving JSON | file=/kaggle/working/qfid_matched_stage1_nih_only/execution_events.json
[+ 16818.3s] [OK     ] [2.2] Saved JSON | file=execution_events.json | size_kb=469.1 | elapsed=0.02s
[+ 16818.3s] [SAVE   ] [2.2] Saving JSON | file=/kaggle/working/qfid_matched_stage1_nih_only/run_status.json
[+ 16818.3s] [OK     ] [2.2] Saved JSON | file=run_status.json | size_kb=0.3 | elapsed=0.00s
[+ 16818.3s] [STATUS ] [2.3] Run status updated | status=complete | next=None
[+ 16818.3s] [SUCCESS] [15.6] STAGE 1 NIH-ONLY COMPLETE | output=/kaggle/working/qfid_matched_stage1_nih_only | required_artifacts=13/13 | total_elapsed=16818.3s | gpu0_alloc_mb=1392.1 | gpu0_reserved_mb=5474.0 | gpu1_alloc_mb=18.2 | gpu1_reserved_mb=3530.0
[+ 16818.3s] [CELL   ] [15.6] END CELL — Final artifact-completeness gate | elapsed=0.03s


## 16. Output contract for downstream notebooks

Load `QFID_MATCHED_STAGE1_BUNDLE.pt`. It contains the NIH-only whole-image and patch checkpoints plus the unchanged frozen CheXzero reference provenance. The separate `chexzero_matched_nih_cache.pt` contains frozen reference features for the same official NIH cohort.

There is no CheXpert target pre-adaptation or T1 checkpoint in this version.
